<a href="https://colab.research.google.com/github/Misha-private/Demo-repo/blob/main/Hybrid_emulator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
import os
def is_drive_mounted():
    return os.path.exists('/content/drive')
if not is_drive_mounted(): drive.mount('/content/drive')


Mounted at /content/drive


# Original FD model

In [ ]:
# ============================================================
# run_fd_1layer_catalog.py
# ------------------------------------------------------------
# Run 1-layer shallow-water FD model on twisted/Klein beta-plane
# for every IC in a C-grid bundle.
#
# INPUT:
#   /content/drive/MyDrive/klein_ics_1L/bundle_cgrid_1L.npz
#
# OUTPUT:
#   /content/drive/MyDrive/klein_ckpt_1L/<IC_KEY>/klein_step_XXXXXX.npz
#   plus plots and time series
# ============================================================

import os
import numpy as np
import matplotlib.pyplot as plt

# -------------------------
# Paths
# -------------------------
IC_BUNDLE = "/content/drive/MyDrive/AI_4DVAR/klein_ics_1L/bundle_cgrid_1L.npz"
ROOT_OUT  = "/content/drive/MyDrive/AI_4DVAR/klein_ckpt_1L"

# -------------------------
# Physics / grid
# -------------------------
g = 9.81
H = 1000.0
nx = 256
ny = 128
Lx = 2.0e7
Ly = 8.0e6
dx = Lx / nx
dy = Ly / ny
fp = 8.0e-5

dt = 30.0
nt = 1200
SAVE_STEPS = {0, 200, 400, 600, 800, 1000, 1200}

# diffusion
nu2_u, nu2_v, nu2_h = 1.0e4, 1.0e4, 5.0e3
nu4_u, nu4_v, nu4_h = 5.0e10, 5.0e10, 2.5e10

HMIN = 0.5

# -------------------------
# Geometry
# -------------------------
x_c = np.linspace(0.5*dx, Lx-0.5*dx, nx)
y_c = np.linspace(0.5*dy, Ly-0.5*dy, ny)
Xc, Yc = np.meshgrid(x_c, y_c)
phi_c = np.pi*((Yc/Ly) - 0.5)
f_c = fp*np.sin(phi_c)

x_u = np.linspace(0.0, Lx, nx+1)
y_v = np.linspace(0.0, Ly, ny+1)
Xu, Yu = np.meshgrid(x_u, y_c)
Xv, Yv = np.meshgrid(x_c, y_v)
phi_u = np.pi*((Yu/Ly) - 0.5)
phi_v = np.pi*((Yv/Ly) - 0.5)
f_u = fp*np.sin(phi_u)
f_v = fp*np.sin(phi_v)

# -------------------------
# Klein twist BCs
# -------------------------
def twist_reflect_x(arr):
    return arr[..., ::-1]

def apply_bc_1l(u, v, h):
    # centers even
    h[0, :]  = 0.5*(h[1, :]  + twist_reflect_x(h[1, :]))
    h[-1, :] = 0.5*(h[-2, :] + twist_reflect_x(h[-2, :]))
    # u odd
    u[0, :]  = 0.5*(u[1, :]  - twist_reflect_x(u[1, :]))
    u[-1, :] = 0.5*(u[-2, :] - twist_reflect_x(u[-2, :]))
    # v even
    v[0, :]  = 0.5*(v[1, :]  + twist_reflect_x(v[1, :]))
    v[-1, :] = 0.5*(v[-2, :] + twist_reflect_x(v[-2, :]))
    return u, v, h

# -------------------------
# C-grid helpers
# -------------------------
def center_from_u(u):
    return 0.5*(u[:, :-1] + u[:, 1:])

def center_from_v(v):
    return 0.5*(v[:-1, :] + v[1:, :])

def avg_x(a):
    return 0.5*(np.pad(a, ((0,0),(1,0)), mode='wrap') + np.pad(a, ((0,0),(0,1)), mode='wrap'))

def avg_y(a):
    return 0.5*(np.pad(a, ((1,0),(0,0)), mode='edge') + np.pad(a, ((0,1),(0,0)), mode='edge'))

def ddx_c_to_u(phi):
    L = np.pad(phi, ((0,0),(1,0)), mode='wrap')
    R = np.pad(phi, ((0,0),(0,1)), mode='wrap')
    return (R - L) / (2.0*dx)

def ddy_c_to_v(phi):
    T = np.pad(phi, ((1,0),(0,0)), mode='edge')
    B = np.pad(phi, ((0,1),(0,0)), mode='edge')
    return (B - T) / (2.0*dy)

def ddx_u_to_c(phi_u):
    return (phi_u[:, 1:] - phi_u[:, :-1]) / dx

def ddy_v_to_c(phi_v):
    return (phi_v[1:, :] - phi_v[:-1, :]) / dy

# -------------------------
# Laplacians
# -------------------------
def lap_u(u):
    ue = np.pad(u, ((0,0),(1,1)), mode='wrap')
    u_xx = (ue[:, :-2] - 2*ue[:, 1:-1] + ue[:, 2:]) / dx**2
    ue2 = np.pad(u, ((1,1),(0,0)), mode='edge')
    u_yy = (ue2[:-2, :] - 2*ue2[1:-1, :] + ue2[2:, :]) / dy**2
    return u_xx + u_yy

def lap_v(v):
    ve = np.pad(v, ((0,0),(1,1)), mode='wrap')
    v_xx = (ve[:, :-2] - 2*ve[:, 1:-1] + ve[:, 2:]) / dx**2
    ve2 = np.pad(v, ((1,1),(0,0)), mode='edge')
    v_yy = (ve2[:-2, :] - 2*ve2[1:-1, :] + ve2[2:, :]) / dy**2
    return v_xx + v_yy

def lap_c(h):
    he = np.pad(h, ((0,0),(1,1)), mode='wrap')
    h_xx = (he[:, :-2] - 2*he[:, 1:-1] + he[:, 2:]) / dx**2
    he2 = np.pad(h, ((1,1),(0,0)), mode='edge')
    h_yy = (he2[:-2, :] - 2*he2[1:-1, :] + he2[2:, :]) / dy**2
    return h_xx + h_yy

def bih_u(u): return lap_u(lap_u(u))
def bih_v(v): return lap_v(lap_v(v))
def bih_c(h): return lap_c(lap_c(h))

# -------------------------
# Vorticity
# -------------------------
def compute_corner_vort(u, v):
    v_w = np.pad(v, ((0,0),(1,0)), mode='wrap')
    v_e = np.pad(v, ((0,0),(0,1)), mode='wrap')
    dv_dx = (v_e - v_w) / (2*dx)

    u_s = np.pad(u, ((1,0),(0,0)), mode='edge')
    u_n = np.pad(u, ((0,1),(0,0)), mode='edge')
    du_dy = (u_n - u_s) / (2*dy)

    return dv_dx - du_dy

def to_u_from_corners(a):
    return 0.5*(a[:-1, :] + a[1:, :])

def to_v_from_corners(a):
    return 0.5*(a[:, :-1] + a[:, 1:])

# -------------------------
# Positivity floor
# -------------------------
def enforce_floor_ke_preserving(u, v, h, hmin=HMIN):
    mask = (h < hmin)
    if not np.any(mask):
        return u, v, h, 0

    s_c = np.ones_like(h, dtype=np.float32)
    s_c[mask] = np.sqrt(np.maximum(h[mask], 1e-12) / hmin)

    s_u = 0.5*(np.pad(s_c, ((0,0),(1,0)), mode='wrap') + np.pad(s_c, ((0,0),(0,1)), mode='wrap'))
    s_v = 0.5*(np.pad(s_c, ((1,0),(0,0)), mode='edge') + np.pad(s_c, ((0,1),(0,0)), mode='edge'))

    u = u * s_u
    v = v * s_v
    h = np.maximum(h, hmin)
    return u, v, h, int(mask.sum())

# -------------------------
# RHS (1-layer AL-style)
# -------------------------
def rhs_1l(u, v, h):
    u, v, h = apply_bc_1l(u, v, h)

    uc = center_from_u(u)
    vc = center_from_v(v)
    K = 0.5*(uc**2 + vc**2)

    eta = h - H
    Phi = g * eta

    dPhidx_u = ddx_c_to_u(Phi)
    dPhidy_v = ddy_c_to_v(Phi)
    dKdx_u   = ddx_c_to_u(K)
    dKdy_v   = ddy_c_to_v(K)

    z_corners = compute_corner_vort(u, v)
    eta_u = to_u_from_corners(z_corners) + f_u
    eta_v = to_v_from_corners(z_corners) + f_v

    v_tu = avg_x(center_from_v(v))
    u_tv = avg_y(center_from_u(u))

    du = -(dPhidx_u + dKdx_u) + eta_u * v_tu + nu2_u*lap_u(u) + nu4_u*bih_u(u)
    dv = -(dPhidy_v + dKdy_v) - eta_v * u_tv + nu2_v*lap_v(v) + nu4_v*bih_v(v)

    h_u = avg_x(h)
    h_v = avg_y(h)
    F_u = h_u * u
    F_v = h_v * v

    dhdt = -(ddx_u_to_c(F_u) + ddy_v_to_c(F_v)) + nu2_h*lap_c(h) + nu4_h*bih_c(h)

    return apply_bc_1l(du, dv, dhdt)

# -------------------------
# RK4
# -------------------------
def rk4_1l(u, v, h, dt):
    k1u, k1v, k1h = rhs_1l(u, v, h)

    ub, vb, hb = apply_bc_1l(u + 0.5*dt*k1u, v + 0.5*dt*k1v, h + 0.5*dt*k1h)
    k2u, k2v, k2h = rhs_1l(ub, vb, hb)

    uc, vc, hc = apply_bc_1l(u + 0.5*dt*k2u, v + 0.5*dt*k2v, h + 0.5*dt*k2h)
    k3u, k3v, k3h = rhs_1l(uc, vc, hc)

    ud, vd, hd = apply_bc_1l(u + dt*k3u, v + dt*k3v, h + dt*k3h)
    k4u, k4v, k4h = rhs_1l(ud, vd, hd)

    u_new = u + (dt/6.0)*(k1u + 2*k2u + 2*k3u + k4u)
    v_new = v + (dt/6.0)*(k1v + 2*k2v + 2*k3v + k4v)
    h_new = h + (dt/6.0)*(k1h + 2*k2h + 2*k3h + k4h)

    return apply_bc_1l(u_new, v_new, h_new)

# -------------------------
# Diagnostics / save
# -------------------------
def total_mass(h):
    return float(np.sum(h) * dx * dy)

def total_energy(u, v, h):
    uc = center_from_u(u)
    vc = center_from_v(v)
    eta = h - H
    ke = 0.5*h*(uc**2 + vc**2)
    pe = 0.5*g*(eta**2)
    return float(np.sum((ke + pe) * dx * dy))

def save_centered_1L(ic_key, step, u, v, h, t):
    ic_dir = os.path.join(ROOT_OUT, ic_key)
    os.makedirs(ic_dir, exist_ok=True)

    eta = h - H
    uc = center_from_u(u)
    vc = center_from_v(v)

    path = os.path.join(ic_dir, f"klein_step_{step:06d}.npz")
    np.savez_compressed(
        path,
        eta=eta.astype(np.float32),
        uc=uc.astype(np.float32),
        vc=vc.astype(np.float32),
        h=h.astype(np.float32),
        f=f_c.astype(np.float32),
        y_m=y_c.astype(np.float32),
        H=np.float32(H),
        dt=np.float32(dt),
        t=np.float32(t),
        nx=np.int32(nx), ny=np.int32(ny),
        dx=np.float32(dx), dy=np.float32(dy), fp=np.float32(fp),
    )
    return path

def quick_plot_1L(ic_key, step, u, v, h):
    ic_dir = os.path.join(ROOT_OUT, ic_key)
    pdir = os.path.join(ic_dir, "plots")
    os.makedirs(pdir, exist_ok=True)

    eta = h - H
    uc = center_from_u(u)
    vc = center_from_v(v)

    xkm = Xc / 1e3
    ykm = (Yc - 0.5*Ly) / 1e3

    fig, axs = plt.subplots(1, 3, figsize=(14, 3.5))
    im = axs[0].pcolormesh(xkm, ykm, eta, shading="auto")
    axs[0].set_title("eta (m)")
    plt.colorbar(im, ax=axs[0])

    im = axs[1].pcolormesh(xkm, ykm, uc, shading="auto")
    axs[1].set_title("u_c (m/s)")
    plt.colorbar(im, ax=axs[1])

    im = axs[2].pcolormesh(xkm, ykm, vc, shading="auto")
    axs[2].set_title("v_c (m/s)")
    plt.colorbar(im, ax=axs[2])

    plt.tight_layout()
    plt.savefig(os.path.join(pdir, f"fields_step_{step:06d}.png"), dpi=120)
    plt.close()

def plot_mass_energy(ic_key, steps, m_series, e_series):
    ic_dir = os.path.join(ROOT_OUT, ic_key)
    pdir = os.path.join(ic_dir, "plots")
    os.makedirs(pdir, exist_ok=True)

    tdays = np.asarray(steps) * dt / 86400.0
    M0 = m_series[0]
    E0 = e_series[0]

    plt.figure(figsize=(8,4))
    plt.plot(tdays, (np.asarray(m_series)-M0)/M0, label="ΔM/M0")
    plt.plot(tdays, (np.asarray(e_series)-E0)/E0, label="ΔE/E0")
    plt.xlabel("time (days)")
    plt.ylabel("relative change")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(pdir, "mass_energy_timeseries.png"), dpi=120)
    plt.close()

# -------------------------
# Load bundle and IC keys
# -------------------------
def list_ic_keys(bundle_path):
    d = np.load(bundle_path)
    keys = sorted(set(k[:-2] for k in d.files if k.endswith("_h")))
    return keys

def load_ic(bundle_path, ic_key):
    d = np.load(bundle_path)
    h = d[f"{ic_key}_h"].astype(np.float32)
    u = d[f"{ic_key}_u"].astype(np.float32)
    v = d[f"{ic_key}_v"].astype(np.float32)
    return h, u, v

# -------------------------
# Run one IC
# -------------------------
def run_ic(ic_key):
    print(f"\n=== 1L IC: {ic_key} | nx={nx}, ny={ny}, dt={dt:.1f}s, nt={nt} ===")
    h, u, v = load_ic(IC_BUNDLE, ic_key)
    u, v, h = apply_bc_1l(u, v, h)
    h[:] = np.maximum(h, 5.0)

    steps = []
    m_series = []
    e_series = []

    M0 = total_mass(h)
    E0 = total_energy(u, v, h)
    t = 0.0

    steps.append(0)
    m_series.append(M0)
    e_series.append(E0)

    if 0 in SAVE_STEPS:
        p = save_centered_1L(ic_key, 0, u, v, h, t)
        quick_plot_1L(ic_key, 0, u, v, h)
        print(f"[save] {ic_key} step=0 -> {p}")

    for n in range(1, nt+1):
        u, v, h = rk4_1l(u, v, h, dt)
        t += dt

        u, v, h, _ = enforce_floor_ke_preserving(u, v, h, HMIN)

        steps.append(n)
        m_series.append(total_mass(h))
        e_series.append(total_energy(u, v, h))

        if n in SAVE_STEPS:
            p = save_centered_1L(ic_key, n, u, v, h, t)
            quick_plot_1L(ic_key, n, u, v, h)
            print(f"[save] {ic_key} step={n} -> {p}")

        if (n % 100) == 0 or n == 1:
            uc = center_from_u(u)
            vc = center_from_v(v)
            umax = float(np.max(np.sqrt(uc*uc + vc*vc)))
            dM = (m_series[-1] - M0) / M0
            dE = (e_series[-1] - E0) / E0
            print(f"[{n:5d}] dM/M0={dM:+.3e}  dE/E0={dE:+.3e}  umax={umax:6.2f}")

    plot_mass_energy(ic_key, steps, m_series, e_series)
    print(f"Done (1L): {ic_key}")

# -------------------------
# Main
# -------------------------
if __name__ == "__main__":
    os.makedirs(ROOT_OUT, exist_ok=True)
    keys = list_ic_keys(IC_BUNDLE)
    print("ICs in bundle:", keys)
    for k in keys:
        run_ic(k)


ICs in bundle: ['gauss_00', 'gauss_01', 'gauss_02', 'gauss_03', 'gauss_04', 'gauss_05', 'mix_00', 'mix_01', 'mix_02', 'mix_03', 'mix_04', 'mix_05', 'test_modon_00', 'test_modon_01', 'test_rh_00', 'test_rh_01', 'vort_00', 'vort_01', 'vort_02', 'vort_03', 'vort_04', 'vort_05', 'wave_00', 'wave_01', 'wave_02', 'wave_03', 'wave_04', 'wave_05']

=== 1L IC: gauss_00 | nx=256, ny=128, dt=30.0s, nt=1200 ===
[save] gauss_00 step=0 -> /content/drive/MyDrive/AI_4DVAR/klein_ckpt_1L/gauss_00/klein_step_000000.npz
[    1] dM/M0=-6.563e-09  dE/E0=-1.336e-06  umax=  3.36
[  100] dM/M0=+1.280e-06  dE/E0=+1.054e-02  umax=  3.31
[save] gauss_00 step=200 -> /content/drive/MyDrive/AI_4DVAR/klein_ckpt_1L/gauss_00/klein_step_000200.npz
[  200] dM/M0=+5.043e-06  dE/E0=+3.638e-02  umax=  3.17
[  300] dM/M0=+1.121e-05  dE/E0=+6.995e-02  umax=  3.00
[save] gauss_00 step=400 -> /content/drive/MyDrive/AI_4DVAR/klein_ckpt_1L/gauss_00/klein_step_000400.npz
[  400] dM/M0=+1.970e-05  dE/E0=+1.074e-01  umax=  2.91
[  5

# Modified FD model wiith outputs at colocation points

In [ ]:
# ============================================================
# run_fd_1layer_catalog_with_colloc.py
# ------------------------------------------------------------
# Run 1-layer shallow-water FD model on twisted/Klein beta-plane
# for every IC in a C-grid bundle.
#
# INPUT:
#   /content/drive/MyDrive/AI_4DVAR/klein_ics_1L/bundle_cgrid_1L.npz
#
# OUTPUT:
#   /content/drive/MyDrive/AI_4DVAR/klein_ckpt_1L/<IC_KEY>/klein_step_XXXXXX.npz
#   plus plots and time series
#
# NEW OUTPUT:
#   /content/drive/MyDrive/AI_4DVAR/klein_ml_1L/colloc_raw/<IC_KEY>_colloc.npz
#
# The collocation file stores isolated centered grid points plus local FD
# diagnostics at those points:
#   t_sec, fd_step, macro_index, ix, iy, x_m, y_m
#   eta, uc, vc, f
#   deta_dt_fd, duc_dt_fd, dvc_dt_fd
#   etax_fd, etay_fd
#   ucx_fd, ucy_fd
#   vcx_fd, vcy_fd
#   div_fd, zeta_fd
# ============================================================

import os
import numpy as np
import matplotlib.pyplot as plt

# -------------------------
# Paths
# -------------------------
IC_BUNDLE = "/content/drive/MyDrive/AI_4DVAR/klein_ics_1L/bundle_cgrid_1L.npz"
ROOT_OUT  = "/content/drive/MyDrive/AI_4DVAR/klein_ckpt_1L"
ROOT_COLLOC = "/content/drive/MyDrive/AI_4DVAR/klein_ml_1L/colloc_raw"

# -------------------------
# Physics / grid
# -------------------------
g = 9.81
H = 1000.0
nx = 256
ny = 128
Lx = 2.0e7
Ly = 8.0e6
dx = Lx / nx
dy = Ly / ny
fp = 8.0e-5

dt = 30.0
nt = 1200
SAVE_STEPS = {0, 200, 400, 600, 800, 1000, 1200}

# diffusion
nu2_u, nu2_v, nu2_h = 1.0e4, 1.0e4, 5.0e3
nu4_u, nu4_v, nu4_h = 5.0e10, 5.0e10, 2.5e10

HMIN = 0.5

# -------------------------
# Collocation config
# -------------------------
SAVE_COLLOC = True
COLLOC_EVERY_N_STEPS = 20
N_COLLOC_PER_TIME = 256
COLLOC_WEIGHT_MODE = "grad_eta"   # "uniform" or "grad_eta"
COLLOC_SEED = 42
SAVE_COLLOC_COORDS = True

# -------------------------
# Geometry
# -------------------------
x_c = np.linspace(0.5 * dx, Lx - 0.5 * dx, nx)
y_c = np.linspace(0.5 * dy, Ly - 0.5 * dy, ny)
Xc, Yc = np.meshgrid(x_c, y_c)

phi_c = np.pi * ((Yc / Ly) - 0.5)
f_c = fp * np.sin(phi_c)

x_u = np.linspace(0.0, Lx, nx + 1)
y_v = np.linspace(0.0, Ly, ny + 1)
Xu, Yu = np.meshgrid(x_u, y_c)
Xv, Yv = np.meshgrid(x_c, y_v)

phi_u = np.pi * ((Yu / Ly) - 0.5)
phi_v = np.pi * ((Yv / Ly) - 0.5)
f_u = fp * np.sin(phi_u)
f_v = fp * np.sin(phi_v)

# -------------------------
# Klein twist BCs
# -------------------------
def twist_reflect_x(arr):
    return arr[..., ::-1]

def apply_bc_1l(u, v, h):
    # centers even
    h[0, :]  = 0.5 * (h[1, :]  + twist_reflect_x(h[1, :]))
    h[-1, :] = 0.5 * (h[-2, :] + twist_reflect_x(h[-2, :]))

    # u odd
    u[0, :]  = 0.5 * (u[1, :]  - twist_reflect_x(u[1, :]))
    u[-1, :] = 0.5 * (u[-2, :] - twist_reflect_x(u[-2, :]))

    # v even
    v[0, :]  = 0.5 * (v[1, :]  + twist_reflect_x(v[1, :]))
    v[-1, :] = 0.5 * (v[-2, :] + twist_reflect_x(v[-2, :]))
    return u, v, h

# -------------------------
# C-grid helpers
# -------------------------
def center_from_u(u):
    return 0.5 * (u[:, :-1] + u[:, 1:])

def center_from_v(v):
    return 0.5 * (v[:-1, :] + v[1:, :])

def avg_x(a):
    return 0.5 * (
        np.pad(a, ((0, 0), (1, 0)), mode="wrap")
        + np.pad(a, ((0, 0), (0, 1)), mode="wrap")
    )

def avg_y(a):
    return 0.5 * (
        np.pad(a, ((1, 0), (0, 0)), mode="edge")
        + np.pad(a, ((0, 1), (0, 0)), mode="edge")
    )

def ddx_c_to_u(phi):
    L = np.pad(phi, ((0, 0), (1, 0)), mode="wrap")
    R = np.pad(phi, ((0, 0), (0, 1)), mode="wrap")
    return (R - L) / (2.0 * dx)

def ddy_c_to_v(phi):
    T = np.pad(phi, ((1, 0), (0, 0)), mode="edge")
    B = np.pad(phi, ((0, 1), (0, 0)), mode="edge")
    return (B - T) / (2.0 * dy)

def ddx_u_to_c(phi_u):
    return (phi_u[:, 1:] - phi_u[:, :-1]) / dx

def ddy_v_to_c(phi_v):
    return (phi_v[1:, :] - phi_v[:-1, :]) / dy

# -------------------------
# Centered FD diagnostics on emulator variables
# Used only for saved collocation diagnostics, not solver evolution.
# -------------------------
def ddx_center(a):
    return (np.roll(a, -1, axis=1) - np.roll(a, 1, axis=1)) / (2.0 * dx)

def ddy_center_edge(a):
    ap = np.pad(a, ((1, 1), (0, 0)), mode="edge")
    return (ap[2:, :] - ap[:-2, :]) / (2.0 * dy)

# -------------------------
# Laplacians
# -------------------------
def lap_u(u):
    ue = np.pad(u, ((0, 0), (1, 1)), mode="wrap")
    u_xx = (ue[:, :-2] - 2 * ue[:, 1:-1] + ue[:, 2:]) / dx**2

    ue2 = np.pad(u, ((1, 1), (0, 0)), mode="edge")
    u_yy = (ue2[:-2, :] - 2 * ue2[1:-1, :] + ue2[2:, :]) / dy**2
    return u_xx + u_yy

def lap_v(v):
    ve = np.pad(v, ((0, 0), (1, 1)), mode="wrap")
    v_xx = (ve[:, :-2] - 2 * ve[:, 1:-1] + ve[:, 2:]) / dx**2

    ve2 = np.pad(v, ((1, 1), (0, 0)), mode="edge")
    v_yy = (ve2[:-2, :] - 2 * ve2[1:-1, :] + ve2[2:, :]) / dy**2
    return v_xx + v_yy

def lap_c(h):
    he = np.pad(h, ((0, 0), (1, 1)), mode="wrap")
    h_xx = (he[:, :-2] - 2 * he[:, 1:-1] + he[:, 2:]) / dx**2

    he2 = np.pad(h, ((1, 1), (0, 0)), mode="edge")
    h_yy = (he2[:-2, :] - 2 * he2[1:-1, :] + he2[2:, :]) / dy**2
    return h_xx + h_yy

def bih_u(u):
    return lap_u(lap_u(u))

def bih_v(v):
    return lap_v(lap_v(v))

def bih_c(h):
    return lap_c(lap_c(h))

# -------------------------
# Vorticity
# -------------------------
def compute_corner_vort(u, v):
    v_w = np.pad(v, ((0, 0), (1, 0)), mode="wrap")
    v_e = np.pad(v, ((0, 0), (0, 1)), mode="wrap")
    dv_dx = (v_e - v_w) / (2 * dx)

    u_s = np.pad(u, ((1, 0), (0, 0)), mode="edge")
    u_n = np.pad(u, ((0, 1), (0, 0)), mode="edge")
    du_dy = (u_n - u_s) / (2 * dy)

    return dv_dx - du_dy

def to_u_from_corners(a):
    return 0.5 * (a[:-1, :] + a[1:, :])

def to_v_from_corners(a):
    return 0.5 * (a[:, :-1] + a[:, 1:])

# -------------------------
# Positivity floor
# -------------------------
def enforce_floor_ke_preserving(u, v, h, hmin=HMIN):
    mask = (h < hmin)
    if not np.any(mask):
        return u, v, h, 0

    s_c = np.ones_like(h, dtype=np.float32)
    s_c[mask] = np.sqrt(np.maximum(h[mask], 1e-12) / hmin)

    s_u = 0.5 * (
        np.pad(s_c, ((0, 0), (1, 0)), mode="wrap")
        + np.pad(s_c, ((0, 0), (0, 1)), mode="wrap")
    )
    s_v = 0.5 * (
        np.pad(s_c, ((1, 0), (0, 0)), mode="edge")
        + np.pad(s_c, ((0, 1), (0, 0)), mode="edge")
    )

    u = u * s_u
    v = v * s_v
    h = np.maximum(h, hmin)
    return u, v, h, int(mask.sum())

# -------------------------
# RHS (1-layer AL-style)
# -------------------------
def rhs_1l(u, v, h):
    u, v, h = apply_bc_1l(u, v, h)

    uc = center_from_u(u)
    vc = center_from_v(v)
    K = 0.5 * (uc**2 + vc**2)

    eta = h - H
    Phi = g * eta

    dPhidx_u = ddx_c_to_u(Phi)
    dPhidy_v = ddy_c_to_v(Phi)
    dKdx_u   = ddx_c_to_u(K)
    dKdy_v   = ddy_c_to_v(K)

    z_corners = compute_corner_vort(u, v)
    eta_u = to_u_from_corners(z_corners) + f_u
    eta_v = to_v_from_corners(z_corners) + f_v

    v_tu = avg_x(center_from_v(v))
    u_tv = avg_y(center_from_u(u))

    du = -(dPhidx_u + dKdx_u) + eta_u * v_tu + nu2_u * lap_u(u) + nu4_u * bih_u(u)
    dv = -(dPhidy_v + dKdy_v) - eta_v * u_tv + nu2_v * lap_v(v) + nu4_v * bih_v(v)

    h_u = avg_x(h)
    h_v = avg_y(h)
    F_u = h_u * u
    F_v = h_v * v

    dhdt = -(ddx_u_to_c(F_u) + ddy_v_to_c(F_v)) + nu2_h * lap_c(h) + nu4_h * bih_c(h)

    return apply_bc_1l(du, dv, dhdt)

# -------------------------
# RK4
# -------------------------
def rk4_1l(u, v, h, dt):
    k1u, k1v, k1h = rhs_1l(u, v, h)

    ub, vb, hb = apply_bc_1l(u + 0.5 * dt * k1u, v + 0.5 * dt * k1v, h + 0.5 * dt * k1h)
    k2u, k2v, k2h = rhs_1l(ub, vb, hb)

    uc, vc, hc = apply_bc_1l(u + 0.5 * dt * k2u, v + 0.5 * dt * k2v, h + 0.5 * dt * k2h)
    k3u, k3v, k3h = rhs_1l(uc, vc, hc)

    ud, vd, hd = apply_bc_1l(u + dt * k3u, v + dt * k3v, h + dt * k3h)
    k4u, k4v, k4h = rhs_1l(ud, vd, hd)

    u_new = u + (dt / 6.0) * (k1u + 2 * k2u + 2 * k3u + k4u)
    v_new = v + (dt / 6.0) * (k1v + 2 * k2v + 2 * k3v + k4v)
    h_new = h + (dt / 6.0) * (k1h + 2 * k2h + 2 * k3h + k4h)

    return apply_bc_1l(u_new, v_new, h_new)

# -------------------------
# Collocation helpers
# -------------------------
def init_colloc_store():
    return {
        "t_sec": [],
        "fd_step": [],
        "macro_index": [],
        "ix": [],
        "iy": [],
        "x_m": [],
        "y_m": [],
        "eta": [],
        "uc": [],
        "vc": [],
        "f": [],
        "deta_dt_fd": [],
        "duc_dt_fd": [],
        "dvc_dt_fd": [],
        "etax_fd": [],
        "etay_fd": [],
        "ucx_fd": [],
        "ucy_fd": [],
        "vcx_fd": [],
        "vcy_fd": [],
        "div_fd": [],
        "zeta_fd": [],
    }

def finalize_colloc_store(store):
    out = {}
    for k, v in store.items():
        if len(v) == 0:
            if k in ("fd_step", "macro_index", "ix", "iy"):
                out[k] = np.array([], dtype=np.int32)
            else:
                out[k] = np.array([], dtype=np.float32)
        else:
            out[k] = np.concatenate(v, axis=0)
    return out

def save_colloc_npz(ic_key, store):
    os.makedirs(ROOT_COLLOC, exist_ok=True)
    out_path = os.path.join(ROOT_COLLOC, f"{ic_key}_colloc.npz")

    save_dict = finalize_colloc_store(store)
    save_dict["ic_key"] = np.array(ic_key)
    save_dict["nx"] = np.int32(nx)
    save_dict["ny"] = np.int32(ny)
    save_dict["dx"] = np.float32(dx)
    save_dict["dy"] = np.float32(dy)
    save_dict["dt"] = np.float32(dt)
    save_dict["H"] = np.float32(H)
    save_dict["fp"] = np.float32(fp)

    np.savez_compressed(out_path, **save_dict)
    return out_path

def sample_collocation_indices(weight, npts, rng):
    flat_n = ny * nx
    npts = min(npts, flat_n)

    if weight is None:
        idx = rng.choice(flat_n, size=npts, replace=False)
    else:
        w = np.asarray(weight, dtype=np.float64).reshape(-1)
        w = np.maximum(w, 0.0)
        s = w.sum()
        if s <= 0.0:
            idx = rng.choice(flat_n, size=npts, replace=False)
        else:
            w = w / s
            idx = rng.choice(flat_n, size=npts, replace=False, p=w)

    iy = idx // nx
    ix = idx % nx
    return iy.astype(np.int32), ix.astype(np.int32)

def colloc_weight_from_eta(eta, mode="grad_eta"):
    if mode == "uniform":
        return None
    if mode == "grad_eta":
        ex = ddx_center(eta)
        ey = ddy_center_edge(eta)
        w = np.sqrt(ex * ex + ey * ey)
        return w + 1.0e-12
    return None

def get_centered_state_and_fd_diagnostics(u, v, h):
    """
    Return centered state, centered FD time tendencies, and local centered
    FD spatial diagnostics, all on the (ny, nx) centered grid.

    State:
        eta, uc, vc

    Time tendencies:
        deta_dt_fd, duc_dt_fd, dvc_dt_fd

    Spatial diagnostics:
        etax_fd, etay_fd
        ucx_fd, ucy_fd
        vcx_fd, vcy_fd
        div_fd, zeta_fd
    """
    u, v, h = apply_bc_1l(u, v, h)

    eta = h - H
    uc = center_from_u(u)
    vc = center_from_v(v)

    du, dv, dhdt = rhs_1l(u, v, h)

    deta_dt_fd = dhdt
    duc_dt_fd = center_from_u(du)
    dvc_dt_fd = center_from_v(dv)

    etax_fd = ddx_center(eta)
    etay_fd = ddy_center_edge(eta)

    ucx_fd = ddx_center(uc)
    ucy_fd = ddy_center_edge(uc)

    vcx_fd = ddx_center(vc)
    vcy_fd = ddy_center_edge(vc)

    div_fd = ucx_fd + vcy_fd
    zeta_fd = vcx_fd - ucy_fd

    return (
        eta, uc, vc,
        deta_dt_fd, duc_dt_fd, dvc_dt_fd,
        etax_fd, etay_fd,
        ucx_fd, ucy_fd,
        vcx_fd, vcy_fd,
        div_fd, zeta_fd,
    )

def append_colloc_samples(store, step, macro_index, t, u, v, h, rng):
    (
        eta, uc, vc,
        deta_dt_fd, duc_dt_fd, dvc_dt_fd,
        etax_fd, etay_fd,
        ucx_fd, ucy_fd,
        vcx_fd, vcy_fd,
        div_fd, zeta_fd,
    ) = get_centered_state_and_fd_diagnostics(u, v, h)

    weight = colloc_weight_from_eta(eta, mode=COLLOC_WEIGHT_MODE)
    iy, ix = sample_collocation_indices(weight, N_COLLOC_PER_TIME, rng)

    store["t_sec"].append(np.full(len(iy), t, dtype=np.float32))
    store["fd_step"].append(np.full(len(iy), step, dtype=np.int32))
    store["macro_index"].append(np.full(len(iy), macro_index, dtype=np.int32))

    store["ix"].append(ix)
    store["iy"].append(iy)

    if SAVE_COLLOC_COORDS:
        store["x_m"].append(Xc[iy, ix].astype(np.float32))
        store["y_m"].append(Yc[iy, ix].astype(np.float32))
    else:
        store["x_m"].append(np.zeros(len(iy), dtype=np.float32))
        store["y_m"].append(np.zeros(len(iy), dtype=np.float32))

    store["eta"].append(eta[iy, ix].astype(np.float32))
    store["uc"].append(uc[iy, ix].astype(np.float32))
    store["vc"].append(vc[iy, ix].astype(np.float32))
    store["f"].append(f_c[iy, ix].astype(np.float32))

    store["deta_dt_fd"].append(deta_dt_fd[iy, ix].astype(np.float32))
    store["duc_dt_fd"].append(duc_dt_fd[iy, ix].astype(np.float32))
    store["dvc_dt_fd"].append(dvc_dt_fd[iy, ix].astype(np.float32))

    store["etax_fd"].append(etax_fd[iy, ix].astype(np.float32))
    store["etay_fd"].append(etay_fd[iy, ix].astype(np.float32))

    store["ucx_fd"].append(ucx_fd[iy, ix].astype(np.float32))
    store["ucy_fd"].append(ucy_fd[iy, ix].astype(np.float32))

    store["vcx_fd"].append(vcx_fd[iy, ix].astype(np.float32))
    store["vcy_fd"].append(vcy_fd[iy, ix].astype(np.float32))

    store["div_fd"].append(div_fd[iy, ix].astype(np.float32))
    store["zeta_fd"].append(zeta_fd[iy, ix].astype(np.float32))

# -------------------------
# Diagnostics / save
# -------------------------
def total_mass(h):
    return float(np.sum(h) * dx * dy)

def total_energy(u, v, h):
    uc = center_from_u(u)
    vc = center_from_v(v)
    eta = h - H
    ke = 0.5 * h * (uc**2 + vc**2)
    pe = 0.5 * g * (eta**2)
    return float(np.sum((ke + pe) * dx * dy))

def save_centered_1L(ic_key, step, u, v, h, t):
    ic_dir = os.path.join(ROOT_OUT, ic_key)
    os.makedirs(ic_dir, exist_ok=True)

    eta = h - H
    uc = center_from_u(u)
    vc = center_from_v(v)

    path = os.path.join(ic_dir, f"klein_step_{step:06d}.npz")
    np.savez_compressed(
        path,
        eta=eta.astype(np.float32),
        uc=uc.astype(np.float32),
        vc=vc.astype(np.float32),
        h=h.astype(np.float32),
        f=f_c.astype(np.float32),
        y_m=y_c.astype(np.float32),
        H=np.float32(H),
        dt=np.float32(dt),
        t=np.float32(t),
        nx=np.int32(nx), ny=np.int32(ny),
        dx=np.float32(dx), dy=np.float32(dy), fp=np.float32(fp),
    )
    return path

def quick_plot_1L(ic_key, step, u, v, h):
    ic_dir = os.path.join(ROOT_OUT, ic_key)
    pdir = os.path.join(ic_dir, "plots")
    os.makedirs(pdir, exist_ok=True)

    eta = h - H
    uc = center_from_u(u)
    vc = center_from_v(v)

    xkm = Xc / 1e3
    ykm = (Yc - 0.5 * Ly) / 1e3

    fig, axs = plt.subplots(1, 3, figsize=(14, 3.5))

    im = axs[0].pcolormesh(xkm, ykm, eta, shading="auto")
    axs[0].set_title("eta (m)")
    plt.colorbar(im, ax=axs[0])

    im = axs[1].pcolormesh(xkm, ykm, uc, shading="auto")
    axs[1].set_title("u_c (m/s)")
    plt.colorbar(im, ax=axs[1])

    im = axs[2].pcolormesh(xkm, ykm, vc, shading="auto")
    axs[2].set_title("v_c (m/s)")
    plt.colorbar(im, ax=axs[2])

    plt.tight_layout()
    plt.savefig(os.path.join(pdir, f"fields_step_{step:06d}.png"), dpi=120)
    plt.close()

def plot_mass_energy(ic_key, steps, m_series, e_series):
    ic_dir = os.path.join(ROOT_OUT, ic_key)
    pdir = os.path.join(ic_dir, "plots")
    os.makedirs(pdir, exist_ok=True)

    tdays = np.asarray(steps) * dt / 86400.0
    M0 = m_series[0]
    E0 = e_series[0]

    plt.figure(figsize=(8, 4))
    plt.plot(tdays, (np.asarray(m_series) - M0) / M0, label="ΔM/M0")
    plt.plot(tdays, (np.asarray(e_series) - E0) / E0, label="ΔE/E0")
    plt.xlabel("time (days)")
    plt.ylabel("relative change")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(pdir, "mass_energy_timeseries.png"), dpi=120)
    plt.close()

# -------------------------
# Load bundle and IC keys
# -------------------------
def list_ic_keys(bundle_path):
    d = np.load(bundle_path)
    keys = sorted(set(k[:-2] for k in d.files if k.endswith("_h")))
    return keys

def load_ic(bundle_path, ic_key):
    d = np.load(bundle_path)
    h = d[f"{ic_key}_h"].astype(np.float32)
    u = d[f"{ic_key}_u"].astype(np.float32)
    v = d[f"{ic_key}_v"].astype(np.float32)
    return h, u, v

# -------------------------
# Run one IC
# -------------------------
def run_ic(ic_key):
    print(f"\n=== 1L IC: {ic_key} | nx={nx}, ny={ny}, dt={dt:.1f}s, nt={nt} ===")

    h, u, v = load_ic(IC_BUNDLE, ic_key)
    u, v, h = apply_bc_1l(u, v, h)
    h[:] = np.maximum(h, 5.0)

    steps = []
    m_series = []
    e_series = []

    M0 = total_mass(h)
    E0 = total_energy(u, v, h)
    t = 0.0

    rng = np.random.default_rng(COLLOC_SEED)
    colloc_store = init_colloc_store()
    macro_index = 0

    steps.append(0)
    m_series.append(M0)
    e_series.append(E0)

    if 0 in SAVE_STEPS:
        p = save_centered_1L(ic_key, 0, u, v, h, t)
        quick_plot_1L(ic_key, 0, u, v, h)
        print(f"[save] {ic_key} step=0 -> {p}")
        macro_index += 1

    if SAVE_COLLOC and (0 % COLLOC_EVERY_N_STEPS == 0):
        append_colloc_samples(
            store=colloc_store,
            step=0,
            macro_index=macro_index,
            t=t,
            u=u, v=v, h=h,
            rng=rng,
        )

    for n in range(1, nt + 1):
        u, v, h = rk4_1l(u, v, h, dt)
        t += dt

        u, v, h, _ = enforce_floor_ke_preserving(u, v, h, HMIN)

        steps.append(n)
        m_series.append(total_mass(h))
        e_series.append(total_energy(u, v, h))

        if n in SAVE_STEPS:
            p = save_centered_1L(ic_key, n, u, v, h, t)
            quick_plot_1L(ic_key, n, u, v, h)
            print(f"[save] {ic_key} step={n} -> {p}")
            macro_index += 1

        if SAVE_COLLOC and (n % COLLOC_EVERY_N_STEPS == 0):
            append_colloc_samples(
                store=colloc_store,
                step=n,
                macro_index=macro_index,
                t=t,
                u=u, v=v, h=h,
                rng=rng,
            )

        if (n % 100) == 0 or n == 1:
            uc = center_from_u(u)
            vc = center_from_v(v)
            umax = float(np.max(np.sqrt(uc * uc + vc * vc)))
            dM = (m_series[-1] - M0) / M0
            dE = (e_series[-1] - E0) / E0
            print(f"[{n:5d}] dM/M0={dM:+.3e}  dE/E0={dE:+.3e}  umax={umax:6.2f}")

    plot_mass_energy(ic_key, steps, m_series, e_series)

    if SAVE_COLLOC:
        colloc_path = save_colloc_npz(ic_key, colloc_store)
        print(f"[colloc] {ic_key} -> {colloc_path}")

    print(f"Done (1L): {ic_key}")

# -------------------------
# Main
# -------------------------
if __name__ == "__main__":
    os.makedirs(ROOT_OUT, exist_ok=True)
    os.makedirs(ROOT_COLLOC, exist_ok=True)

    keys = list_ic_keys(IC_BUNDLE)
    print("ICs in bundle:", keys)
    for k in keys:
        run_ic(k)

ICs in bundle: ['gauss_00', 'gauss_01', 'gauss_02', 'gauss_03', 'gauss_04', 'gauss_05', 'mix_00', 'mix_01', 'mix_02', 'mix_03', 'mix_04', 'mix_05', 'test_modon_00', 'test_modon_01', 'test_rh_00', 'test_rh_01', 'vort_00', 'vort_01', 'vort_02', 'vort_03', 'vort_04', 'vort_05', 'wave_00', 'wave_01', 'wave_02', 'wave_03', 'wave_04', 'wave_05']

=== 1L IC: gauss_00 | nx=256, ny=128, dt=30.0s, nt=1200 ===
[save] gauss_00 step=0 -> /content/drive/MyDrive/AI_4DVAR/klein_ckpt_1L/gauss_00/klein_step_000000.npz
[    1] dM/M0=-6.563e-09  dE/E0=-1.336e-06  umax=  3.36
[  100] dM/M0=+1.280e-06  dE/E0=+1.054e-02  umax=  3.31
[save] gauss_00 step=200 -> /content/drive/MyDrive/AI_4DVAR/klein_ckpt_1L/gauss_00/klein_step_000200.npz
[  200] dM/M0=+5.043e-06  dE/E0=+3.638e-02  umax=  3.17
[  300] dM/M0=+1.121e-05  dE/E0=+6.995e-02  umax=  3.00
[save] gauss_00 step=400 -> /content/drive/MyDrive/AI_4DVAR/klein_ckpt_1L/gauss_00/klein_step_000400.npz
[  400] dM/M0=+1.970e-05  dE/E0=+1.074e-01  umax=  2.91
[  5

# Test collocation files

In [ ]:
# ============================================================
# inspect_colloc_samples_1L.py
# ------------------------------------------------------------
# Inspect several *_colloc.npz files produced by
# run_fd_1layer_catalog_with_colloc.py
#
# Checks:
#   - required arrays exist
#   - lengths match
#   - no NaN / Inf
#   - ranges look reasonable
#   - sample counts match expectation
#   - optional plots for several cases
#
# Colab-friendly.
# ============================================================

import os
import glob
import numpy as np
import matplotlib.pyplot as plt

# ============================================================
# USER SETTINGS
# ============================================================

ROOT_COLLOC = "/content/drive/MyDrive/AI_4DVAR/klein_ml_1L/colloc_raw"
ROOT_SNAP   = "/content/drive/MyDrive/AI_4DVAR/klein_ckpt_1L"

# How many cases to inspect
MAX_CASES = 5

# If None, auto-pick first MAX_CASES files
SELECT_CASES = None
# Example:
# SELECT_CASES = ["rh4", "modon_00", "wave_00"]

# Expected run settings used when collocation files were created
NT_EXPECTED = 1200
COLLOC_EVERY_N_STEPS_EXPECTED = 20
N_COLLOC_PER_TIME_EXPECTED = 256

# Plot controls
MAKE_PLOTS = True
SAVE_PLOTS = True
PLOT_DIR = "/content/drive/MyDrive/AI_4DVAR/klein_ml_1L/colloc_inspection_plots"

# Histogram bins
NBINS = 60

# ============================================================
# REQUIRED ARRAYS
# ============================================================

REQUIRED_KEYS = [
    "t_sec", "fd_step", "macro_index",
    "ix", "iy", "x_m", "y_m",
    "eta", "uc", "vc", "f",
    "deta_dt_fd", "duc_dt_fd", "dvc_dt_fd",
    "etax_fd", "etay_fd",
    "ucx_fd", "ucy_fd",
    "vcx_fd", "vcy_fd",
    "div_fd", "zeta_fd",
]

SUMMARY_KEYS = [
    "eta", "uc", "vc",
    "deta_dt_fd", "duc_dt_fd", "dvc_dt_fd",
    "etax_fd", "etay_fd",
    "div_fd", "zeta_fd",
]

# ============================================================
# HELPERS
# ============================================================

def find_colloc_files(root_colloc, select_cases=None, max_cases=5):
    files = sorted(glob.glob(os.path.join(root_colloc, "*_colloc.npz")))
    if select_cases is None:
        return files[:max_cases]

    selected = []
    wanted = set(select_cases)
    for fp in files:
        name = os.path.basename(fp).replace("_colloc.npz", "")
        if name in wanted:
            selected.append(fp)
    return selected[:max_cases]


def expected_sample_count(nt, every_n, n_per_time):
    # includes step 0
    n_times = nt // every_n + 1
    return n_times * n_per_time


def print_header(title):
    print("\n" + "=" * 80)
    print(title)
    print("=" * 80)


def safe_stats(arr):
    arr = np.asarray(arr)
    finite = np.isfinite(arr)
    n_total = arr.size
    n_finite = int(finite.sum())
    n_bad = n_total - n_finite

    if n_finite == 0:
        return {
            "size": n_total,
            "finite": n_finite,
            "bad": n_bad,
            "min": np.nan,
            "max": np.nan,
            "mean": np.nan,
            "std": np.nan,
        }

    good = arr[finite]
    return {
        "size": n_total,
        "finite": n_finite,
        "bad": n_bad,
        "min": float(np.min(good)),
        "max": float(np.max(good)),
        "mean": float(np.mean(good)),
        "std": float(np.std(good)),
    }


def print_stats(name, arr):
    s = safe_stats(arr)
    print(
        f"{name:12s} "
        f"size={s['size']:8d} "
        f"bad={s['bad']:6d} "
        f"min={s['min']:+.4e} "
        f"max={s['max']:+.4e} "
        f"mean={s['mean']:+.4e} "
        f"std={s['std']:+.4e}"
    )


def validate_required_keys(z, required_keys):
    missing = [k for k in required_keys if k not in z.files]
    return missing


def validate_lengths(z, keys):
    lengths = {}
    for k in keys:
        lengths[k] = len(z[k])
    unique_lengths = sorted(set(lengths.values()))
    ok = (len(unique_lengths) == 1)
    return ok, lengths, unique_lengths


def summarize_macro_indices(macro_index):
    vals, counts = np.unique(macro_index, return_counts=True)
    return vals, counts


def summarize_fd_steps(fd_step):
    vals, counts = np.unique(fd_step, return_counts=True)
    return vals, counts


def load_snapshot_for_case(ic_key, root_snap):
    snap_files = sorted(glob.glob(os.path.join(root_snap, ic_key, "klein_step_*.npz")))
    if not snap_files:
        return None, None
    # use the first snapshot for quick background plotting
    z = np.load(snap_files[0])
    return z, snap_files[0]


def make_case_plots(ic_key, zc, root_snap, plot_dir):
    os.makedirs(plot_dir, exist_ok=True)

    t_sec = zc["t_sec"]
    eta = zc["eta"]
    uc = zc["uc"]
    vc = zc["vc"]
    deta_dt_fd = zc["deta_dt_fd"]
    div_fd = zc["div_fd"]
    zeta_fd = zc["zeta_fd"]
    x_m = zc["x_m"]
    y_m = zc["y_m"]

    # --------------------------------------------------------
    # 1. Histogram panel
    # --------------------------------------------------------
    fig, axs = plt.subplots(2, 3, figsize=(13, 7))

    axs[0, 0].hist(eta, bins=NBINS)
    axs[0, 0].set_title("eta")

    axs[0, 1].hist(uc, bins=NBINS)
    axs[0, 1].set_title("uc")

    axs[0, 2].hist(vc, bins=NBINS)
    axs[0, 2].set_title("vc")

    axs[1, 0].hist(deta_dt_fd, bins=NBINS)
    axs[1, 0].set_title("deta_dt_fd")

    axs[1, 1].hist(div_fd, bins=NBINS)
    axs[1, 1].set_title("div_fd")

    axs[1, 2].hist(zeta_fd, bins=NBINS)
    axs[1, 2].set_title("zeta_fd")

    plt.tight_layout()
    out1 = os.path.join(plot_dir, f"{ic_key}_histograms.png")
    plt.savefig(out1, dpi=140, bbox_inches="tight")
    plt.close(fig)

    # --------------------------------------------------------
    # 2. Time histogram
    # --------------------------------------------------------
    plt.figure(figsize=(8, 4))
    plt.hist(t_sec / 3600.0, bins=NBINS)
    plt.xlabel("time (hours)")
    plt.ylabel("count")
    plt.title(f"{ic_key}: collocation sample times")
    plt.tight_layout()
    out2 = os.path.join(plot_dir, f"{ic_key}_time_hist.png")
    plt.savefig(out2, dpi=140, bbox_inches="tight")
    plt.close()

    # --------------------------------------------------------
    # 3. Scatter of collocation points over first saved eta field if available
    # --------------------------------------------------------
    zs, snap_path = load_snapshot_for_case(ic_key, root_snap)
    if zs is not None:
        eta_bg = zs["eta"]
        nx = int(zs["nx"])
        ny = int(zs["ny"])
        dx = float(zs["dx"])
        dy = float(zs["dy"])

        x_c = np.linspace(0.5 * dx, nx * dx - 0.5 * dx, nx) / 1e3
        y_c = (np.linspace(0.5 * dy, ny * dy - 0.5 * dy, ny) - 0.5 * ny * dy) / 1e3

        Xc, Yc = np.meshgrid(x_c, y_c)

        plt.figure(figsize=(8, 4))
        plt.pcolormesh(Xc, Yc, eta_bg, shading="auto")
        plt.scatter(x_m / 1e3, (y_m - 0.5 * ny * dy) / 1e3, s=5, alpha=0.35)
        plt.xlabel("x (km)")
        plt.ylabel("y (km)")
        plt.title(f"{ic_key}: collocation points over eta background")
        plt.tight_layout()
        out3 = os.path.join(plot_dir, f"{ic_key}_points_over_eta.png")
        plt.savefig(out3, dpi=140, bbox_inches="tight")
        plt.close()

    return


def inspect_one_file(fp):
    ic_key = os.path.basename(fp).replace("_colloc.npz", "")
    print_header(f"Inspecting {ic_key}")

    z = np.load(fp, allow_pickle=True)

    # 1) Required keys
    missing = validate_required_keys(z, REQUIRED_KEYS)
    if missing:
        print("Missing keys:")
        for k in missing:
            print("   ", k)
    else:
        print("All required keys are present.")

    # 2) Length checks
    keys_present = [k for k in REQUIRED_KEYS if k in z.files]
    ok_len, lengths, unique_lengths = validate_lengths(z, keys_present)
    if ok_len:
        n_samples = unique_lengths[0]
        print(f"All required arrays have matching length: {n_samples}")
    else:
        print("Length mismatch detected:")
        for k, v in lengths.items():
            print(f"  {k:12s}: {v}")
        n_samples = min(unique_lengths)

    # 3) Expected sample count
    expected = expected_sample_count(
        NT_EXPECTED,
        COLLOC_EVERY_N_STEPS_EXPECTED,
        N_COLLOC_PER_TIME_EXPECTED,
    )
    print(f"Expected sample count (based on current settings): {expected}")
    print(f"Actual sample count: {n_samples}")

    # 4) Basic metadata
    if "nx" in z.files and "ny" in z.files:
        print(f"Grid: nx={int(z['nx'])}, ny={int(z['ny'])}")
    if "dx" in z.files and "dy" in z.files:
        print(f"dx={float(z['dx']):.3f}, dy={float(z['dy']):.3f}")
    if "dt" in z.files:
        print(f"dt={float(z['dt']):.3f}")

    # 5) Index range checks
    ix = z["ix"]
    iy = z["iy"]
    nx_meta = int(z["nx"]) if "nx" in z.files else None
    ny_meta = int(z["ny"]) if "ny" in z.files else None

    if nx_meta is not None and ny_meta is not None:
        bad_ix = np.sum((ix < 0) | (ix >= nx_meta))
        bad_iy = np.sum((iy < 0) | (iy >= ny_meta))
        print(f"Bad ix count: {bad_ix}")
        print(f"Bad iy count: {bad_iy}")

    # 6) Stats for important fields
    print("\nKey variable statistics:")
    for k in SUMMARY_KEYS:
        if k in z.files:
            print_stats(k, z[k])

    # 7) Step/time summaries
    fd_step = z["fd_step"]
    t_sec = z["t_sec"]
    macro_index = z["macro_index"]

    print("\nTime / step summary:")
    print(f"fd_step min/max: {fd_step.min()} / {fd_step.max()}")
    print(f"t_sec   min/max: {t_sec.min():.2f} / {t_sec.max():.2f}")
    print(f"macro_index min/max: {macro_index.min()} / {macro_index.max()}")

    vals_s, counts_s = summarize_fd_steps(fd_step)
    print(f"Unique fd_step count: {len(vals_s)}")
    if len(vals_s) <= 20:
        print("fd_step counts:")
        for v, c in zip(vals_s, counts_s):
            print(f"  step={int(v):5d}  count={int(c):6d}")

    vals_m, counts_m = summarize_macro_indices(macro_index)
    print(f"Unique macro_index count: {len(vals_m)}")
    if len(vals_m) <= 20:
        print("macro_index counts:")
        for v, c in zip(vals_m, counts_m):
            print(f"  macro={int(v):5d}  count={int(c):6d}")

    # 8) NaN/Inf report
    print("\nNaN/Inf report:")
    for k in SUMMARY_KEYS:
        if k in z.files:
            bad = np.sum(~np.isfinite(z[k]))
            print(f"{k:12s} bad={int(bad)}")

    return z, ic_key


# ============================================================
# MAIN
# ============================================================

def main():
    os.makedirs(PLOT_DIR, exist_ok=True)

    files = find_colloc_files(ROOT_COLLOC, SELECT_CASES, MAX_CASES)
    print_header("Collocation inspection")
    print(f"Directory: {ROOT_COLLOC}")
    print(f"Files selected: {len(files)}")

    if len(files) == 0:
        print("No collocation files found.")
        return

    for fp in files:
        z, ic_key = inspect_one_file(fp)
        if MAKE_PLOTS:
            make_case_plots(ic_key, z, ROOT_SNAP, PLOT_DIR)
            print(f"Saved plots for {ic_key} in: {PLOT_DIR}")

    print_header("Done")


if __name__ == "__main__":
    main()


Collocation inspection
Directory: /content/drive/MyDrive/AI_4DVAR/klein_ml_1L/colloc_raw
Files selected: 5

Inspecting gauss_00
All required keys are present.
All required arrays have matching length: 15616
Expected sample count (based on current settings): 15616
Actual sample count: 15616
Grid: nx=256, ny=128
dx=78125.000, dy=62500.000
dt=30.000
Bad ix count: 0
Bad iy count: 0

Key variable statistics:
eta          size=   15616 bad=     0 min=-9.9899e+00 max=+3.3961e+01 mean=+6.1146e+00 std=+8.5172e+00
uc           size=   15616 bad=     0 min=-3.3293e+00 max=+1.6427e+00 mean=-2.3961e-01 std=+9.3980e-01
vc           size=   15616 bad=     0 min=-2.5304e+00 max=+2.5826e+00 mean=+2.1798e-02 std=+1.0550e+00
deta_dt_fd   size=   15616 bad=     0 min=-3.4895e-03 max=+3.4116e-03 mean=+1.1544e-04 std=+5.8736e-04
duc_dt_fd    size=   15616 bad=     0 min=-6.3324e-05 max=+1.0803e-04 mean=+5.0928e-06 std=+2.7581e-05
dvc_dt_fd    size=   15616 bad=     0 min=-1.9748e-04 max=+1.8728e-04 mean=+3

# Build collocation loader

In [ ]:
# ============================================================
# colloc_bank.py
# ------------------------------------------------------------
# Utility for loading and sampling collocation files produced by
# run_fd_1layer_catalog_with_colloc.py
#
# Main use:
#   from colloc_bank import CollocBank
#
#   bank = CollocBank(COLLOC_DIR)
#   batch = bank.sample_by_macro_index("rh4", macro_index=3, npts=64)
#
# Returned fields are NumPy arrays for the selected collocation points.
#
# This is designed to be simple, explicit, and Colab-friendly.
# ============================================================

import os
import glob
import numpy as np


class CollocBank:
    """
    Load and organize collocation files.

    Each file is expected to be:
        <ic_key>_colloc.npz

    with arrays such as:
        t_sec, fd_step, macro_index,
        ix, iy, x_m, y_m,
        eta, uc, vc, f,
        deta_dt_fd, duc_dt_fd, dvc_dt_fd,
        etax_fd, etay_fd,
        ucx_fd, ucy_fd,
        vcx_fd, vcy_fd,
        div_fd, zeta_fd

    The class builds lookups so training code can quickly sample points
    associated with a chosen IC and time grouping.
    """

    REQUIRED_KEYS = [
        "t_sec", "fd_step", "macro_index",
        "ix", "iy", "x_m", "y_m",
        "eta", "uc", "vc", "f",
        "deta_dt_fd", "duc_dt_fd", "dvc_dt_fd",
        "etax_fd", "etay_fd",
        "ucx_fd", "ucy_fd",
        "vcx_fd", "vcy_fd",
        "div_fd", "zeta_fd",
    ]

    META_KEYS = ["ic_key", "nx", "ny", "dx", "dy", "dt", "H", "fp"]

    def __init__(self, colloc_dir, file_pattern="*_colloc.npz", seed=1234, verbose=True):
        """
        Parameters
        ----------
        colloc_dir : str
            Directory containing collocation .npz files.
        file_pattern : str
            Glob pattern for collocation files.
        seed : int
            RNG seed for reproducible sampling.
        verbose : bool
            Whether to print summary information.
        """
        self.colloc_dir = colloc_dir
        self.file_pattern = file_pattern
        self.verbose = verbose
        self.rng = np.random.default_rng(seed)

        # Main storage:
        #   self.data_by_ic[ic_key] = {field_name: np.ndarray, ...}
        self.data_by_ic = {}

        # Per-IC metadata:
        #   self.meta_by_ic[ic_key] = {...}
        self.meta_by_ic = {}

        # Lookup tables:
        #   self.lookup_by_step[ic_key][fd_step] = np.ndarray(indices)
        #   self.lookup_by_macro[ic_key][macro_index] = np.ndarray(indices)
        self.lookup_by_step = {}
        self.lookup_by_macro = {}

        self._load_all_files()

    # --------------------------------------------------------
    # Internal loading
    # --------------------------------------------------------
    def _load_all_files(self):
        files = sorted(glob.glob(os.path.join(self.colloc_dir, self.file_pattern)))
        if self.verbose:
            print(f"[CollocBank] scanning: {self.colloc_dir}")
            print(f"[CollocBank] found {len(files)} collocation files")

        if len(files) == 0:
            raise FileNotFoundError(
                f"No collocation files found in {self.colloc_dir} "
                f"with pattern {self.file_pattern}"
            )

        for fp in files:
            self._load_one_file(fp)

        if self.verbose:
            self.summary()

    def _load_one_file(self, fp):
        z = np.load(fp, allow_pickle=True)

        # Infer IC key
        if "ic_key" in z.files:
            ic_key = str(z["ic_key"])
        else:
            ic_key = os.path.basename(fp).replace("_colloc.npz", "")

        # Validate required fields
        missing = [k for k in self.REQUIRED_KEYS if k not in z.files]
        if missing:
            raise ValueError(f"{fp} is missing required keys: {missing}")

        # Load required fields
        data = {k: z[k] for k in self.REQUIRED_KEYS}

        # Check lengths match
        lengths = [len(data[k]) for k in self.REQUIRED_KEYS]
        if len(set(lengths)) != 1:
            msg = ", ".join(f"{k}:{len(data[k])}" for k in self.REQUIRED_KEYS)
            raise ValueError(f"{fp} has mismatched array lengths -> {msg}")

        # Load metadata if available
        meta = {}
        for k in self.META_KEYS:
            if k in z.files:
                meta[k] = z[k]

        self.data_by_ic[ic_key] = data
        self.meta_by_ic[ic_key] = meta

        # Build lookups
        step_lookup = {}
        macro_lookup = {}

        fd_step = data["fd_step"]
        macro_index = data["macro_index"]

        for s in np.unique(fd_step):
            step_lookup[int(s)] = np.where(fd_step == s)[0]

        for m in np.unique(macro_index):
            macro_lookup[int(m)] = np.where(macro_index == m)[0]

        self.lookup_by_step[ic_key] = step_lookup
        self.lookup_by_macro[ic_key] = macro_lookup

        if self.verbose:
            print(
                f"[CollocBank] loaded {ic_key:20s} "
                f"n_samples={len(fd_step):8d} "
                f"n_steps={len(step_lookup):4d} "
                f"n_macro={len(macro_lookup):4d}"
            )

    # --------------------------------------------------------
    # Basic queries
    # --------------------------------------------------------
    def list_ic_keys(self):
        return sorted(self.data_by_ic.keys())

    def has_ic(self, ic_key):
        return ic_key in self.data_by_ic

    def get_meta(self, ic_key):
        self._check_ic(ic_key)
        return self.meta_by_ic[ic_key]

    def get_num_samples(self, ic_key):
        self._check_ic(ic_key)
        return len(self.data_by_ic[ic_key]["fd_step"])

    def available_fd_steps(self, ic_key):
        self._check_ic(ic_key)
        return sorted(self.lookup_by_step[ic_key].keys())

    def available_macro_indices(self, ic_key):
        self._check_ic(ic_key)
        return sorted(self.lookup_by_macro[ic_key].keys())

    def summary(self):
        print("\n[CollocBank] summary")
        print("-" * 60)
        for ic_key in self.list_ic_keys():
            n = self.get_num_samples(ic_key)
            ns = len(self.lookup_by_step[ic_key])
            nm = len(self.lookup_by_macro[ic_key])
            print(f"{ic_key:20s}  n_samples={n:8d}  n_fd_steps={ns:4d}  n_macro={nm:4d}")
        print("-" * 60)

    # --------------------------------------------------------
    # Sampling helpers
    # --------------------------------------------------------
    def sample_by_fd_step(self, ic_key, fd_step, npts, replace=False):
        """
        Sample npts collocation points for a given IC and exact fd_step.
        """
        self._check_ic(ic_key)
        if int(fd_step) not in self.lookup_by_step[ic_key]:
            raise KeyError(f"IC '{ic_key}' has no fd_step={fd_step}")

        idx_all = self.lookup_by_step[ic_key][int(fd_step)]
        idx_sel = self._sample_indices(idx_all, npts, replace=replace)
        return self._extract(ic_key, idx_sel)

    def sample_by_macro_index(self, ic_key, macro_index, npts, replace=False):
        """
        Sample npts collocation points for a given IC and macro_index.
        """
        self._check_ic(ic_key)
        if int(macro_index) not in self.lookup_by_macro[ic_key]:
            raise KeyError(f"IC '{ic_key}' has no macro_index={macro_index}")

        idx_all = self.lookup_by_macro[ic_key][int(macro_index)]
        idx_sel = self._sample_indices(idx_all, npts, replace=replace)
        return self._extract(ic_key, idx_sel)

    def sample_random_anywhere(self, ic_key, npts, replace=False):
        """
        Sample npts collocation points from all samples in one IC.
        """
        self._check_ic(ic_key)
        n_all = self.get_num_samples(ic_key)
        idx_all = np.arange(n_all, dtype=np.int64)
        idx_sel = self._sample_indices(idx_all, npts, replace=replace)
        return self._extract(ic_key, idx_sel)

    def sample_nearest_macro(self, ic_key, macro_index, npts, replace=False):
        """
        Sample from the nearest available macro_index if exact one is missing.
        Useful when training metadata and collocation metadata are not perfectly aligned.
        """
        self._check_ic(ic_key)
        avail = np.array(self.available_macro_indices(ic_key), dtype=np.int64)
        if len(avail) == 0:
            raise ValueError(f"IC '{ic_key}' has no macro indices")

        nearest = int(avail[np.argmin(np.abs(avail - int(macro_index)))])
        idx_all = self.lookup_by_macro[ic_key][nearest]
        idx_sel = self._sample_indices(idx_all, npts, replace=replace)
        out = self._extract(ic_key, idx_sel)
        out["macro_index_requested"] = np.full(len(idx_sel), int(macro_index), dtype=np.int32)
        out["macro_index_used"] = np.full(len(idx_sel), nearest, dtype=np.int32)
        return out

    # --------------------------------------------------------
    # Utility
    # --------------------------------------------------------
    def _check_ic(self, ic_key):
        if ic_key not in self.data_by_ic:
            raise KeyError(f"Unknown ic_key '{ic_key}'. Available: {self.list_ic_keys()}")

    def _sample_indices(self, idx_all, npts, replace=False):
        idx_all = np.asarray(idx_all, dtype=np.int64)
        if len(idx_all) == 0:
            raise ValueError("Cannot sample from an empty index set")

        if replace:
            sel = self.rng.choice(idx_all, size=npts, replace=True)
        else:
            take = min(int(npts), len(idx_all))
            sel = self.rng.choice(idx_all, size=take, replace=False)

        return np.asarray(sel, dtype=np.int64)

    def _extract(self, ic_key, idx_sel):
        data = self.data_by_ic[ic_key]
        out = {k: data[k][idx_sel] for k in self.REQUIRED_KEYS}
        out["ic_key"] = np.array([ic_key] * len(idx_sel))
        return out


# ============================================================
# Small test / demo
# ============================================================
if __name__ == "__main__":
    COLL_DIR = "/content/drive/MyDrive/AI_4DVAR/klein_ml_1L/colloc_raw"

    bank = CollocBank(COLL_DIR, verbose=True)

    keys = bank.list_ic_keys()
    print("\nAvailable IC keys:", keys)

    if len(keys) > 0:
        ic = keys[0]
        print(f"\nTesting with ic_key = {ic}")

        print("Available fd_step values (first 10):", bank.available_fd_steps(ic)[:10])
        print("Available macro_index values:", bank.available_macro_indices(ic))

        # Sample by macro index if available
        macro_vals = bank.available_macro_indices(ic)
        if len(macro_vals) > 0:
            batch = bank.sample_by_macro_index(ic, macro_vals[0], npts=16)
            print("\nSampled by macro_index")
            for k in ["eta", "uc", "vc", "deta_dt_fd", "div_fd", "zeta_fd"]:
                print(f"{k:12s} shape = {batch[k].shape}")

        # Sample by fd step if available
        step_vals = bank.available_fd_steps(ic)
        if len(step_vals) > 0:
            batch2 = bank.sample_by_fd_step(ic, step_vals[0], npts=16)
            print("\nSampled by fd_step")
            for k in ["eta", "uc", "vc", "deta_dt_fd", "div_fd", "zeta_fd"]:
                print(f"{k:12s} shape = {batch2[k].shape}")

        # Random anywhere
        batch3 = bank.sample_random_anywhere(ic, npts=16)
        print("\nRandom sample anywhere")
        for k in ["eta", "uc", "vc", "deta_dt_fd", "div_fd", "zeta_fd"]:
            print(f"{k:12s} shape = {batch3[k].shape}")

    print("\nDone.")

[CollocBank] scanning: /content/drive/MyDrive/AI_4DVAR/klein_ml_1L/colloc_raw
[CollocBank] found 28 collocation files
[CollocBank] loaded gauss_00             n_samples=   15616 n_steps=  61 n_macro=   7
[CollocBank] loaded gauss_01             n_samples=   15616 n_steps=  61 n_macro=   7
[CollocBank] loaded gauss_02             n_samples=   15616 n_steps=  61 n_macro=   7
[CollocBank] loaded gauss_03             n_samples=   15616 n_steps=  61 n_macro=   7
[CollocBank] loaded gauss_04             n_samples=   15616 n_steps=  61 n_macro=   7
[CollocBank] loaded gauss_05             n_samples=   15616 n_steps=  61 n_macro=   7
[CollocBank] loaded mix_00               n_samples=   15616 n_steps=  61 n_macro=   7
[CollocBank] loaded mix_01               n_samples=   15616 n_steps=  61 n_macro=   7
[CollocBank] loaded mix_02               n_samples=   15616 n_steps=  61 n_macro=   7
[CollocBank] loaded mix_03               n_samples=   15616 n_steps=  61 n_macro=   7
[CollocBank] loaded mi

# Predict tendencies with GNN mdoel

In [ ]:
# ============================================================
# train_gnn_1layer_tendency.py
# ------------------------------------------------------------
# First prototype:
#   GNN predicts tendencies instead of next state
#
# No collocation yet — just verify stability
# ============================================================

import os, glob
import numpy as np
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -------------------------
# Paths
# -------------------------
ROOT = "/content/drive/MyDrive/AI_4DVAR"
DATA_DIR = f"{ROOT}/klein_ckpt_1L"

# -------------------------
# Config
# -------------------------
BATCH_SIZE = 1
LR = 2e-3
EPOCHS = 20

ROLL_STEPS = 3
ROLL_GAMMA = 0.9

# IMPORTANT: macro time step
DT_MACRO = 200 * 30.0   # 6000 s

# -------------------------
# Dataset
# -------------------------
class KleinDataset(Dataset):
    def __init__(self, data_dir):
        self.samples = []

        ic_dirs = sorted(glob.glob(os.path.join(data_dir, "*")))
        for ic in ic_dirs:
            files = sorted(glob.glob(os.path.join(ic, "klein_step_*.npz")))
            for i in range(len(files) - 1):
                self.samples.append((files[i], files[i+1]))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        f0, f1 = self.samples[idx]
        z0 = np.load(f0)
        z1 = np.load(f1)

        x = np.stack([z0["eta"], z0["uc"], z0["vc"]], axis=0)
        y = np.stack([z1["eta"], z1["uc"], z1["vc"]], axis=0)

        return torch.tensor(x, dtype=torch.float32), \
               torch.tensor(y, dtype=torch.float32)

# -------------------------
# Model (simple CNN)
# Replace later with GNN
# -------------------------
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 64, 3, padding=1),
            nn.GELU(),
            nn.Conv2d(64, 64, 3, padding=1),
            nn.GELU(),
            nn.Conv2d(64, 3, 3, padding=1),
        )

    def forward(self, x):
        return self.net(x)  # outputs tendencies

# -------------------------
# Rollout loss
# -------------------------
def rollout_loss(model, x0, y_true):
    """
    Multi-step rollout using tendency formulation
    """
    loss = 0.0
    x = x0.clone()

    for k in range(ROLL_STEPS):
        xdot = model(x)
        x = x + DT_MACRO * xdot

        loss_k = ((x - y_true)**2).mean()
        loss += (ROLL_GAMMA**k) * loss_k

    return loss

# -------------------------
# Train
# -------------------------
def train():
    dataset = KleinDataset(DATA_DIR)
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

    model = SimpleCNN().to(device)
    opt = torch.optim.Adam(model.parameters(), lr=LR)

    for epoch in range(EPOCHS):
        model.train()
        total_loss = 0.0

        for x, y in loader:
            x = x.to(device)
            y = y.to(device)

            opt.zero_grad()

            loss = rollout_loss(model, x, y)

            loss.backward()
            opt.step()

            total_loss += loss.item()

        print(f"Epoch {epoch:03d} | Loss = {total_loss:.6e}")

    return model

# -------------------------
if __name__ == "__main__":
    train()

ValueError: num_samples should be a positive integer value, but got num_samples=0

In [ ]:
# ============================================================
# train_gnn_1layer_tendency_safe_autopath.py
# ------------------------------------------------------------
# Safe tendency-based emulator prototype for 1-layer Klein data.
#
# Improvements:
#   - auto-detects snapshot root directory
#   - robust dataset scanning with clear prints
#   - GPU-ready
#   - checkpoint saving
#   - reduced dataset load for debugging
#   - gradient clipping
#
# Still only:
#   predict tendencies -> explicit coarse update
#
# No collocation loss yet.
# ============================================================

import os
import glob
import csv
import time
import random
import numpy as np
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ------------------------------------------------------------
# Device
# ------------------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------
ROOT = "/content/drive/MyDrive/AI_4DVAR"

# Candidate snapshot roots to try automatically
DATA_DIR_CANDIDATES = [
    f"{ROOT}/klein_ckpt_1L",
    f"{ROOT}/klein_ckpt_AL_centers_colloc",
    f"{ROOT}/klein_ckpt_AL_centers",
    f"{ROOT}/klein_ckpt_1L_colloc",
]

CKPT_DIR = f"{ROOT}/klein_ml_1L_ckpt_tendency_safe"
os.makedirs(CKPT_DIR, exist_ok=True)

# ------------------------------------------------------------
# Config
# ------------------------------------------------------------
BATCH_SIZE = 2
LR = 1e-3
EPOCHS = 30
WEIGHT_DECAY = 1e-6
GRAD_CLIP = 1.0

ROLL_STEPS = 2
ROLL_GAMMA = 0.9

# Your saved snapshots are every 200 FD steps:
DT_MACRO = 200.0 * 30.0  # 6000 s

# Use only a few pairs per IC while debugging.
# Set to None later to use all pairs.
MAX_PAIRS_PER_IC = 6

NUM_WORKERS = 0
PIN_MEMORY = torch.cuda.is_available()

SAVE_EVERY_EPOCH = 5
RESUME_FROM = None
PRINT_BATCH_EVERY = 20

# ------------------------------------------------------------
# Helper: detect valid data root
# ------------------------------------------------------------
def count_pairs_in_root(data_dir):
    """
    Count how many valid snapshot pairs exist under:
        data_dir/<IC_KEY>/klein_step_*.npz
    """
    if not os.path.exists(data_dir):
        return 0, 0

    ic_dirs = [d for d in glob.glob(os.path.join(data_dir, "*")) if os.path.isdir(d)]
    n_ic = 0
    n_pairs = 0

    for ic_dir in ic_dirs:
        files = sorted(glob.glob(os.path.join(ic_dir, "klein_step_*.npz")))
        if len(files) >= 2:
            n_ic += 1
            n_pairs += (len(files) - 1)

    return n_ic, n_pairs

def autodetect_data_dir(candidates):
    print("[AutoDetect] checking candidate snapshot roots...")
    best_dir = None
    best_pairs = -1

    for d in candidates:
        n_ic, n_pairs = count_pairs_in_root(d)
        print(f"  {d}")
        print(f"     valid IC dirs = {n_ic}, total pairs = {n_pairs}")
        if n_pairs > best_pairs:
            best_pairs = n_pairs
            best_dir = d

    if best_dir is None or best_pairs <= 0:
        msg = "No valid snapshot root found.\n"
        msg += "Checked these paths:\n"
        for d in candidates:
            msg += f"  - {d}\n"
        msg += "Expected structure:\n"
        msg += "  ROOT/<IC_KEY>/klein_step_XXXXXX.npz\n"
        raise RuntimeError(msg)

    print(f"[AutoDetect] using: {best_dir}")
    return best_dir

# ------------------------------------------------------------
# Dataset
# ------------------------------------------------------------
class KleinDataset(Dataset):
    """
    Loads consecutive saved snapshots:
        klein_step_000000.npz -> klein_step_000200.npz -> ...

    Inputs:
        x = [eta, uc, vc] at time n
    Targets:
        y = [eta, uc, vc] at time n+1
    """
    def __init__(self, data_dir, max_pairs_per_ic=None):
        self.samples = []

        print(f"[Dataset] scanning data_dir: {data_dir}")
        if not os.path.exists(data_dir):
            raise RuntimeError(f"DATA_DIR does not exist: {data_dir}")

        all_entries = sorted(glob.glob(os.path.join(data_dir, "*")))
        ic_dirs = [d for d in all_entries if os.path.isdir(d)]
        print(f"[Dataset] found {len(ic_dirs)} subdirectories")

        for ic_dir in ic_dirs:
            ic_key = os.path.basename(ic_dir)
            files = sorted(glob.glob(os.path.join(ic_dir, "klein_step_*.npz")))
            print(f"[Dataset] {ic_key:20s} -> {len(files)} snapshot files")

            if len(files) < 2:
                continue

            pairs = [(files[i], files[i + 1], ic_key) for i in range(len(files) - 1)]

            if max_pairs_per_ic is not None:
                pairs = pairs[:max_pairs_per_ic]

            self.samples.extend(pairs)

        if len(self.samples) == 0:
            raise RuntimeError(
                f"No training pairs found in {data_dir}.\n"
                f"Expected files like: <IC_KEY>/klein_step_XXXXXX.npz"
            )

        print(f"[Dataset] total training pairs = {len(self.samples)}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        f0, f1, ic_key = self.samples[idx]

        z0 = np.load(f0)
        z1 = np.load(f1)

        x = np.stack([z0["eta"], z0["uc"], z0["vc"]], axis=0).astype(np.float32)
        y = np.stack([z1["eta"], z1["uc"], z1["vc"]], axis=0).astype(np.float32)

        return {
            "x": torch.from_numpy(x),
            "y": torch.from_numpy(y),
            "ic_key": ic_key,
            "f0": f0,
            "f1": f1,
        }

# ------------------------------------------------------------
# Model
# ------------------------------------------------------------
class SimpleCNN(nn.Module):
    """
    Predicts tendencies:
        input  = [eta, uc, vc]
        output = [eta_t, uc_t, vc_t]
    """
    def __init__(self, in_ch=3, hidden=64, out_ch=3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, hidden, kernel_size=3, padding=1),
            nn.GELU(),
            nn.Conv2d(hidden, hidden, kernel_size=3, padding=1),
            nn.GELU(),
            nn.Conv2d(hidden, hidden, kernel_size=3, padding=1),
            nn.GELU(),
            nn.Conv2d(hidden, out_ch, kernel_size=3, padding=1),
        )

    def forward(self, x):
        return self.net(x)

# ------------------------------------------------------------
# Loss
# ------------------------------------------------------------
def one_step_update(x, xdot, dt_macro):
    return x + dt_macro * xdot

def rollout_loss(model, x0, y_true, dt_macro, roll_steps=2, gamma=0.9):
    """
    Safe first prototype rollout.

    Note:
    This dataset currently supplies only one next-state target y_true,
    so internal rollout compares repeatedly to the same next target.
    This is okay as a stability test for the tendency formulation.
    """
    x = x0
    total = 0.0

    for k in range(roll_steps):
        xdot = model(x)
        x = one_step_update(x, xdot, dt_macro)
        loss_k = ((x - y_true) ** 2).mean()
        total = total + (gamma ** k) * loss_k

    return total

# ------------------------------------------------------------
# Checkpoint helpers
# ------------------------------------------------------------
def save_checkpoint(path, model, optimizer, epoch, loss_history, data_dir):
    torch.save(
        {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "loss_history": loss_history,
            "data_dir": data_dir,
        },
        path,
    )

def load_checkpoint(path, model, optimizer=None):
    ckpt = torch.load(path, map_location=device)
    model.load_state_dict(ckpt["model_state_dict"])
    if optimizer is not None and "optimizer_state_dict" in ckpt:
        optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    epoch = ckpt.get("epoch", -1)
    loss_history = ckpt.get("loss_history", [])
    return epoch, loss_history, ckpt

# ------------------------------------------------------------
# Logging
# ------------------------------------------------------------
def save_loss_csv(path, loss_history):
    with open(path, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["epoch", "train_loss"])
        for i, val in enumerate(loss_history):
            writer.writerow([i, val])

# ------------------------------------------------------------
# Train
# ------------------------------------------------------------
def train():
    data_dir = autodetect_data_dir(DATA_DIR_CANDIDATES)

    dataset = KleinDataset(data_dir, max_pairs_per_ic=MAX_PAIRS_PER_IC)
    loader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        drop_last=False,
    )

    model = SimpleCNN().to(device)
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=LR,
        weight_decay=WEIGHT_DECAY,
    )

    start_epoch = 0
    loss_history = []

    if RESUME_FROM is not None and os.path.exists(RESUME_FROM):
        print(f"[Resume] loading checkpoint: {RESUME_FROM}")
        loaded_epoch, loaded_history, ckpt = load_checkpoint(RESUME_FROM, model, optimizer)
        start_epoch = loaded_epoch + 1
        loss_history = loaded_history
        print(f"[Resume] starting from epoch {start_epoch}")
        print(f"[Resume] checkpoint data_dir was: {ckpt.get('data_dir', 'unknown')}")

    print(f"[Train] using data_dir = {data_dir}")
    print(f"[Train] dataset pairs = {len(dataset)}")
    print(f"[Train] batches/epoch = {len(loader)}")

    for epoch in range(start_epoch, EPOCHS):
        t0 = time.time()
        model.train()
        running_loss = 0.0

        for ib, batch in enumerate(loader):
            x = batch["x"].to(device, non_blocking=True)
            y = batch["y"].to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            loss = rollout_loss(
                model=model,
                x0=x,
                y_true=y,
                dt_macro=DT_MACRO,
                roll_steps=ROLL_STEPS,
                gamma=ROLL_GAMMA,
            )

            if not torch.isfinite(loss):
                raise RuntimeError(f"Non-finite loss at epoch={epoch}, batch={ib}: {loss.item()}")

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step()

            running_loss += loss.item()

            if (ib % PRINT_BATCH_EVERY) == 0:
                print(
                    f"[epoch {epoch:03d} | batch {ib:04d}/{len(loader):04d}] "
                    f"loss={loss.item():.6e}"
                )

        epoch_loss = running_loss / max(len(loader), 1)
        loss_history.append(epoch_loss)

        dt_epoch = time.time() - t0
        print(
            f"Epoch {epoch:03d} done | "
            f"train_loss={epoch_loss:.6e} | "
            f"time={dt_epoch:.1f}s"
        )

        # Save rolling checkpoint
        last_ckpt = os.path.join(CKPT_DIR, "last_model.pt")
        save_checkpoint(last_ckpt, model, optimizer, epoch, loss_history, data_dir)

        # Save periodic checkpoint
        if ((epoch + 1) % SAVE_EVERY_EPOCH) == 0:
            epoch_ckpt = os.path.join(CKPT_DIR, f"model_epoch_{epoch:03d}.pt")
            save_checkpoint(epoch_ckpt, model, optimizer, epoch, loss_history, data_dir)

        # Save CSV every epoch
        save_loss_csv(os.path.join(CKPT_DIR, "loss_history.csv"), loss_history)

    final_ckpt = os.path.join(CKPT_DIR, "final_model.pt")
    save_checkpoint(final_ckpt, model, optimizer, EPOCHS - 1, loss_history, data_dir)
    save_loss_csv(os.path.join(CKPT_DIR, "loss_history.csv"), loss_history)

    print("[Train] finished.")
    print(f"[Train] final checkpoint: {final_ckpt}")
    return model, loss_history

# ------------------------------------------------------------
# Main
# ------------------------------------------------------------
if __name__ == "__main__":
    train()

Using device: cuda
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
[AutoDetect] checking candidate snapshot roots...
  /content/drive/MyDrive/AI_4DVAR/klein_ckpt_1L
     valid IC dirs = 28, total pairs = 168
  /content/drive/MyDrive/AI_4DVAR/klein_ckpt_AL_centers_colloc
     valid IC dirs = 0, total pairs = 0
  /content/drive/MyDrive/AI_4DVAR/klein_ckpt_AL_centers
     valid IC dirs = 0, total pairs = 0
  /content/drive/MyDrive/AI_4DVAR/klein_ckpt_1L_colloc
     valid IC dirs = 0, total pairs = 0
[AutoDetect] using: /content/drive/MyDrive/AI_4DVAR/klein_ckpt_1L
[Dataset] scanning data_dir: /content/drive/MyDrive/AI_4DVAR/klein_ckpt_1L
[Dataset] found 28 subdirectories
[Dataset] gauss_00             -> 7 snapshot files
[Dataset] gauss_01             -> 7 snapshot files
[Dataset] gauss_02             -> 7 snapshot files
[Dataset] gauss_03             -> 7 snapshot files
[Dataset] gauss_04             -> 7 snapshot files
[Dataset] gauss_05             -> 7 snapshot files
[Dataset] mix_0

# Create coloc_bank.py

In [ ]:
# ============================================================
# colloc_bank.py
# Simple collocation sampler for shallow-water emulator
# ============================================================

import os
import glob
import numpy as np


class CollocBank:
    SAMPLE_KEYS = [
        "t_sec", "fd_step", "macro_index",
        "ix", "iy", "x_m", "y_m",
        "eta", "uc", "vc", "f",
        "deta_dt_fd", "duc_dt_fd", "dvc_dt_fd",
        "etax_fd", "etay_fd",
        "ucx_fd", "ucy_fd",
        "vcx_fd", "vcy_fd",
        "div_fd", "zeta_fd",
    ]

    META_KEYS = [
        "ic_key", "nx", "ny", "dx", "dy", "dt", "H", "fp"
    ]

    def __init__(self, colloc_dir, verbose=True):
        self.colloc_dir = colloc_dir
        self.verbose = verbose

        self.data_by_ic = {}
        self.lookup_by_macro = {}

        self._load_all()

    def _load_all(self):
        files = sorted(glob.glob(os.path.join(self.colloc_dir, "*_colloc.npz")))

        if len(files) == 0:
            raise RuntimeError(f"No collocation files found in {self.colloc_dir}")

        for fp in files:
            ic_key = os.path.basename(fp).replace("_colloc.npz", "")
            z = np.load(fp, allow_pickle=True)

            # keep the loaded npz object
            self.data_by_ic[ic_key] = z

            if "macro_index" not in z.files:
                raise RuntimeError(f"{fp} missing required field 'macro_index'")

            macro = z["macro_index"]
            lookup = {}
            for m in np.unique(macro):
                lookup[int(m)] = np.where(macro == m)[0]

            self.lookup_by_macro[ic_key] = lookup

            if self.verbose:
                print(f"[CollocBank] loaded {ic_key} | samples={len(macro)} | macro_groups={len(lookup)}")

    def has_ic(self, ic_key):
        return ic_key in self.data_by_ic

    def available_macro_indices(self, ic_key):
        if ic_key not in self.lookup_by_macro:
            return []
        return sorted(self.lookup_by_macro[ic_key].keys())

    def sample_nearest_macro(self, ic_key, macro_index, npts, replace=False):
        if ic_key not in self.data_by_ic:
            raise KeyError(f"IC key not found in colloc bank: {ic_key}")

        z = self.data_by_ic[ic_key]
        lookup = self.lookup_by_macro[ic_key]

        macros = np.array(list(lookup.keys()), dtype=np.int32)
        if len(macros) == 0:
            raise RuntimeError(f"No macro groups for ic={ic_key}")

        nearest = int(macros[np.argmin(np.abs(macros - int(macro_index)))])
        idx_all = lookup[nearest]

        if len(idx_all) == 0:
            raise RuntimeError(f"No samples for ic={ic_key}, macro={nearest}")

        if replace:
            idx = np.random.choice(idx_all, size=npts, replace=True)
        else:
            n = min(npts, len(idx_all))
            idx = np.random.choice(idx_all, size=n, replace=False)

        out = {}

        # index only sample-wise arrays
        for k in self.SAMPLE_KEYS:
            if k in z.files:
                out[k] = z[k][idx]

        # copy metadata unchanged
        for k in self.META_KEYS:
            if k in z.files:
                out[k] = z[k]

        out["macro_index_requested"] = np.int32(macro_index)
        out["macro_index_used"] = np.int32(nearest)

        return out

In [ ]:
from pathlib import Path

path = Path("/content/drive/MyDrive/AI_4DVAR/colloc_bank.py")

code = r'''
# ============================================================
# colloc_bank.py
# Simple collocation sampler for shallow-water emulator
# ============================================================

import os
import glob
import numpy as np


class CollocBank:
    SAMPLE_KEYS = [
        "t_sec", "fd_step", "macro_index",
        "ix", "iy", "x_m", "y_m",
        "eta", "uc", "vc", "f",
        "deta_dt_fd", "duc_dt_fd", "dvc_dt_fd",
        "etax_fd", "etay_fd",
        "ucx_fd", "ucy_fd",
        "vcx_fd", "vcy_fd",
        "div_fd", "zeta_fd",
    ]

    META_KEYS = [
        "ic_key", "nx", "ny", "dx", "dy", "dt", "H", "fp"
    ]

    def __init__(self, colloc_dir, verbose=True):
        self.colloc_dir = colloc_dir
        self.verbose = verbose

        self.data_by_ic = {}
        self.lookup_by_macro = {}

        self._load_all()

    def _load_all(self):
        files = sorted(glob.glob(os.path.join(self.colloc_dir, "*_colloc.npz")))

        if len(files) == 0:
            raise RuntimeError(f"No collocation files found in {self.colloc_dir}")

        for fp in files:
            ic_key = os.path.basename(fp).replace("_colloc.npz", "")
            z = np.load(fp, allow_pickle=True)

            self.data_by_ic[ic_key] = z

            if "macro_index" not in z.files:
                raise RuntimeError(f"{fp} missing required field 'macro_index'")

            macro = z["macro_index"]
            lookup = {}
            for m in np.unique(macro):
                lookup[int(m)] = np.where(macro == m)[0]

            self.lookup_by_macro[ic_key] = lookup

            if self.verbose:
                print(f"[CollocBank] loaded {ic_key} | samples={len(macro)} | macro_groups={len(lookup)}")

    def has_ic(self, ic_key):
        return ic_key in self.data_by_ic

    def available_macro_indices(self, ic_key):
        if ic_key not in self.lookup_by_macro:
            return []
        return sorted(self.lookup_by_macro[ic_key].keys())

    def sample_nearest_macro(self, ic_key, macro_index, npts, replace=False):
        if ic_key not in self.data_by_ic:
            raise KeyError(f"IC key not found in colloc bank: {ic_key}")

        z = self.data_by_ic[ic_key]
        lookup = self.lookup_by_macro[ic_key]

        macros = np.array(list(lookup.keys()), dtype=np.int32)
        if len(macros) == 0:
            raise RuntimeError(f"No macro groups for ic={ic_key}")

        nearest = int(macros[np.argmin(np.abs(macros - int(macro_index)))])
        idx_all = lookup[nearest]

        if len(idx_all) == 0:
            raise RuntimeError(f"No samples for ic={ic_key}, macro={nearest}")

        if replace:
            idx = np.random.choice(idx_all, size=npts, replace=True)
        else:
            n = min(npts, len(idx_all))
            idx = np.random.choice(idx_all, size=n, replace=False)

        out = {}

        for k in self.SAMPLE_KEYS:
            if k in z.files:
                out[k] = z[k][idx]

        for k in self.META_KEYS:
            if k in z.files:
                out[k] = z[k]

        out["macro_index_requested"] = np.int32(macro_index)
        out["macro_index_used"] = np.int32(nearest)

        return out
'''
path.write_text(code)
print("Updated:", path)

Updated: /content/drive/MyDrive/AI_4DVAR/colloc_bank.py


In [ ]:
!ls /content/drive/MyDrive/AI_4DVAR

colloc_bank.py		    klein_ml_1L_ckpt_cnn_multistep
experiments_1L_gnn	    klein_ml_1L_ckpt_cnn_multistep_physics
klein_ckpt_1L		    klein_ml_1L_ckpt_gnn
klein_ckpt_1L_fine20	    klein_ml_1L_ckpt_tendency_colloc
klein_ckpt_1L_fine20_rhaug  klein_ml_1L_ckpt_tendency_safe
klein_ics_1L		    klein_ml_1L_fine
klein_ml_1L		    klein_ml_1L_fine_rhaug
klein_ml_1L_ckpt_cnn	    __pycache__


In [ ]:
import sys
sys.path.append("/content/drive/MyDrive/AI_4DVAR")

from colloc_bank import CollocBank

print("Import OK!")

Import OK!


# Actual training

In [ ]:
# ============================================================
# train_gnn_1layer_tendency_colloc.py
# ------------------------------------------------------------
# First hybrid prototype:
#   - predict tendencies instead of next state
#   - add collocation tendency matching loss
#
# Loss:
#   L = L_data + lambda_colloc * L_colloc_tend
#
# This version is:
#   - GPU-ready
#   - robust to path issues
#   - checkpointed
#   - collocation-aware
#
# NOTE:
#   Uses the existing colloc_bank.py helper.
#   Model is still a simple CNN tendency predictor for now.
# ============================================================

import os
import sys
import glob
import csv
import time
import random
import numpy as np
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

# ------------------------------------------------------------
# Make sure colloc_bank.py can be imported
# ------------------------------------------------------------
sys.path.append(os.getcwd())

from colloc_bank import CollocBank

# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ------------------------------------------------------------
# Device
# ------------------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------
ROOT = "/content/drive/MyDrive/AI_4DVAR"

DATA_DIR_CANDIDATES = [
    f"{ROOT}/klein_ckpt_1L",
    f"{ROOT}/klein_ckpt_AL_centers_colloc",
    f"{ROOT}/klein_ckpt_AL_centers",
    f"{ROOT}/klein_ckpt_1L_colloc",
]

COLLOC_DIR = f"{ROOT}/klein_ml_1L/colloc_raw"
CKPT_DIR = f"{ROOT}/klein_ml_1L_ckpt_tendency_colloc"
os.makedirs(CKPT_DIR, exist_ok=True)

# ------------------------------------------------------------
# Config
# ------------------------------------------------------------
BATCH_SIZE = 2
LR = 1e-3
EPOCHS = 30
WEIGHT_DECAY = 1e-6
GRAD_CLIP = 1.0

ROLL_STEPS = 2
ROLL_GAMMA = 0.9
DT_MACRO = 200.0 * 30.0  # 6000 s

MAX_PAIRS_PER_IC = 6   # set None later for full dataset
NUM_WORKERS = 0
PIN_MEMORY = torch.cuda.is_available()

# Collocation loss
USE_COLLOC = True
N_COLLOC_BATCH = 64
#LAMBDA_COLLOC = 0.05
LAMBDA_COLLOC = 100.0

SAVE_EVERY_EPOCH = 5
RESUME_FROM = None
PRINT_BATCH_EVERY = 20

# ------------------------------------------------------------
# Helper: detect valid data root
# ------------------------------------------------------------
def count_pairs_in_root(data_dir):
    if not os.path.exists(data_dir):
        return 0, 0

    ic_dirs = [d for d in glob.glob(os.path.join(data_dir, "*")) if os.path.isdir(d)]
    n_ic = 0
    n_pairs = 0

    for ic_dir in ic_dirs:
        files = sorted(glob.glob(os.path.join(ic_dir, "klein_step_*.npz")))
        if len(files) >= 2:
            n_ic += 1
            n_pairs += (len(files) - 1)

    return n_ic, n_pairs

def autodetect_data_dir(candidates):
    print("[AutoDetect] checking candidate snapshot roots...")
    best_dir = None
    best_pairs = -1

    for d in candidates:
        n_ic, n_pairs = count_pairs_in_root(d)
        print(f"  {d}")
        print(f"     valid IC dirs = {n_ic}, total pairs = {n_pairs}")
        if n_pairs > best_pairs:
            best_pairs = n_pairs
            best_dir = d

    if best_dir is None or best_pairs <= 0:
        msg = "No valid snapshot root found.\n"
        msg += "Checked these paths:\n"
        for d in candidates:
            msg += f"  - {d}\n"
        msg += "Expected structure:\n"
        msg += "  ROOT/<IC_KEY>/klein_step_XXXXXX.npz\n"
        raise RuntimeError(msg)

    print(f"[AutoDetect] using: {best_dir}")
    return best_dir

# ------------------------------------------------------------
# Dataset
# ------------------------------------------------------------
class KleinDataset(Dataset):
    """
    Loads consecutive snapshot pairs.

    Returns:
      x           : [3, ny, nx]
      y           : [3, ny, nx]
      ic_key      : str
      macro_index : int

    IMPORTANT:
    macro_index here is aligned to the collocation generation script:
      pair index i  -> macro_index = i + 1

    so that the input state at:
      klein_step_000000 -> macro_index 1
      klein_step_000200 -> macro_index 2
      ...
    """
    def __init__(self, data_dir, max_pairs_per_ic=None):
        self.samples = []

        print(f"[Dataset] scanning data_dir: {data_dir}")
        if not os.path.exists(data_dir):
            raise RuntimeError(f"DATA_DIR does not exist: {data_dir}")

        ic_dirs = sorted([d for d in glob.glob(os.path.join(data_dir, "*")) if os.path.isdir(d)])
        print(f"[Dataset] found {len(ic_dirs)} subdirectories")

        for ic_dir in ic_dirs:
            ic_key = os.path.basename(ic_dir)
            files = sorted(glob.glob(os.path.join(ic_dir, "klein_step_*.npz")))
            print(f"[Dataset] {ic_key:20s} -> {len(files)} snapshot files")

            if len(files) < 2:
                continue

            pairs = []
            for i in range(len(files) - 1):
                f0 = files[i]
                f1 = files[i + 1]
                macro_index = i + 1
                pairs.append((f0, f1, ic_key, macro_index))

            if max_pairs_per_ic is not None:
                pairs = pairs[:max_pairs_per_ic]

            self.samples.extend(pairs)

        if len(self.samples) == 0:
            raise RuntimeError(
                f"No training pairs found in {data_dir}.\n"
                f"Expected files like: <IC_KEY>/klein_step_XXXXXX.npz"
            )

        print(f"[Dataset] total training pairs = {len(self.samples)}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        f0, f1, ic_key, macro_index = self.samples[idx]

        z0 = np.load(f0)
        z1 = np.load(f1)

        x = np.stack([z0["eta"], z0["uc"], z0["vc"]], axis=0).astype(np.float32)
        y = np.stack([z1["eta"], z1["uc"], z1["vc"]], axis=0).astype(np.float32)

        return {
            "x": torch.from_numpy(x),
            "y": torch.from_numpy(y),
            "ic_key": ic_key,
            "macro_index": np.int32(macro_index),
            "f0": f0,
            "f1": f1,
        }

# ------------------------------------------------------------
# Model
# ------------------------------------------------------------
class SimpleCNN(nn.Module):
    """
    Predicts tendencies:
        input  = [eta, uc, vc]
        output = [eta_t, uc_t, vc_t]
    """
    def __init__(self, in_ch=3, hidden=64, out_ch=3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, hidden, kernel_size=3, padding=1),
            nn.GELU(),
            nn.Conv2d(hidden, hidden, kernel_size=3, padding=1),
            nn.GELU(),
            nn.Conv2d(hidden, hidden, kernel_size=3, padding=1),
            nn.GELU(),
            nn.Conv2d(hidden, out_ch, kernel_size=3, padding=1),
        )

    def forward(self, x):
        return self.net(x)

# ------------------------------------------------------------
# Loss helpers
# ------------------------------------------------------------
def one_step_update(x, xdot, dt_macro):
    return x + dt_macro * xdot

def rollout_loss(model, x0, y_true, dt_macro, roll_steps=2, gamma=0.9):
    x = x0
    total = 0.0

    for k in range(roll_steps):
        xdot = model(x)
        x = one_step_update(x, xdot, dt_macro)
        loss_k = ((x - y_true) ** 2).mean()
        total = total + (gamma ** k) * loss_k

    return total

def colloc_tendency_loss_for_batch(xdot_pred, batch, colloc_bank, n_colloc_batch):
    """
    Compare predicted grid tendencies against saved FD tendencies at collocation points.

    xdot_pred : torch.Tensor [B, 3, ny, nx]
    batch     : dict from DataLoader, containing ic_key and macro_index

    Returns
    -------
    loss_colloc : torch scalar
    n_used      : number of batch items that actually used collocation samples
    """
    B = xdot_pred.shape[0]
    total = 0.0
    n_used = 0

    ic_keys = batch["ic_key"]
    macro_indices = batch["macro_index"]

    for b in range(B):
        ic_key = ic_keys[b]
        macro_index = int(macro_indices[b])

        if not colloc_bank.has_ic(ic_key):
            continue

        # use nearest macro in case exact one is absent
        colloc = colloc_bank.sample_nearest_macro(
            ic_key=ic_key,
            macro_index=macro_index,
            npts=n_colloc_batch,
            replace=False,
        )

        iy = torch.as_tensor(colloc["iy"], dtype=torch.long, device=xdot_pred.device)
        ix = torch.as_tensor(colloc["ix"], dtype=torch.long, device=xdot_pred.device)

        pred_eta_t = xdot_pred[b, 0, iy, ix]
        pred_uc_t  = xdot_pred[b, 1, iy, ix]
        pred_vc_t  = xdot_pred[b, 2, iy, ix]

        true_eta_t = torch.as_tensor(colloc["deta_dt_fd"], dtype=torch.float32, device=xdot_pred.device)
        true_uc_t  = torch.as_tensor(colloc["duc_dt_fd"],  dtype=torch.float32, device=xdot_pred.device)
        true_vc_t  = torch.as_tensor(colloc["dvc_dt_fd"],  dtype=torch.float32, device=xdot_pred.device)

        loss_b = ((pred_eta_t - true_eta_t) ** 2).mean()
        loss_b = loss_b + ((pred_uc_t - true_uc_t) ** 2).mean()
        loss_b = loss_b + ((pred_vc_t - true_vc_t) ** 2).mean()

        total = total + loss_b
        n_used += 1

    if n_used == 0:
        return torch.tensor(0.0, device=xdot_pred.device), 0

    return total / n_used, n_used

# ------------------------------------------------------------
# Checkpoint helpers
# ------------------------------------------------------------
def save_checkpoint(path, model, optimizer, epoch, loss_history, data_dir):
    torch.save(
        {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "loss_history": loss_history,
            "data_dir": data_dir,
        },
        path,
    )

def load_checkpoint(path, model, optimizer=None):
    ckpt = torch.load(path, map_location=device)
    model.load_state_dict(ckpt["model_state_dict"])
    if optimizer is not None and "optimizer_state_dict" in ckpt:
        optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    epoch = ckpt.get("epoch", -1)
    loss_history = ckpt.get("loss_history", [])
    return epoch, loss_history, ckpt

# ------------------------------------------------------------
# Logging
# ------------------------------------------------------------
def save_loss_csv(path, loss_history):
    with open(path, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["epoch", "train_loss", "train_data_loss", "train_colloc_loss"])
        for row in loss_history:
            writer.writerow(row)

# ------------------------------------------------------------
# Train
# ------------------------------------------------------------
def train():
    data_dir = autodetect_data_dir(DATA_DIR_CANDIDATES)

    if not os.path.exists(COLLOC_DIR):
        raise RuntimeError(f"COLLOC_DIR does not exist: {COLLOC_DIR}")

    colloc_bank = CollocBank(COLLOC_DIR, verbose=True)

    dataset = KleinDataset(data_dir, max_pairs_per_ic=MAX_PAIRS_PER_IC)
    loader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        drop_last=False,
    )

    model = SimpleCNN().to(device)
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=LR,
        weight_decay=WEIGHT_DECAY,
    )

    start_epoch = 0
    loss_history = []

    if RESUME_FROM is not None and os.path.exists(RESUME_FROM):
        print(f"[Resume] loading checkpoint: {RESUME_FROM}")
        loaded_epoch, loaded_history, ckpt = load_checkpoint(RESUME_FROM, model, optimizer)
        start_epoch = loaded_epoch + 1
        loss_history = loaded_history
        print(f"[Resume] starting from epoch {start_epoch}")
        print(f"[Resume] checkpoint data_dir was: {ckpt.get('data_dir', 'unknown')}")

    print(f"[Train] using data_dir = {data_dir}")
    print(f"[Train] using colloc_dir = {COLLOC_DIR}")
    print(f"[Train] dataset pairs = {len(dataset)}")
    print(f"[Train] batches/epoch = {len(loader)}")

    for epoch in range(start_epoch, EPOCHS):
        t0 = time.time()
        model.train()

        running_total = 0.0
        running_data = 0.0
        running_colloc = 0.0
        running_colloc_used = 0

        for ib, batch in enumerate(loader):
            x = batch["x"].to(device, non_blocking=True)
            y = batch["y"].to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            # data loss via tendency rollout
            loss_data = rollout_loss(
                model=model,
                x0=x,
                y_true=y,
                dt_macro=DT_MACRO,
                roll_steps=ROLL_STEPS,
                gamma=ROLL_GAMMA,
            )

            # collocation tendency loss uses the first-step tendency field
            xdot_pred = model(x)

            if USE_COLLOC:
                loss_colloc, n_used = colloc_tendency_loss_for_batch(
                    xdot_pred=xdot_pred,
                    batch=batch,
                    colloc_bank=colloc_bank,
                    n_colloc_batch=N_COLLOC_BATCH,
                )
            else:
                loss_colloc = torch.tensor(0.0, device=device)
                n_used = 0

            loss = loss_data + LAMBDA_COLLOC * loss_colloc

            if not torch.isfinite(loss):
                raise RuntimeError(f"Non-finite loss at epoch={epoch}, batch={ib}: {loss.item()}")

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step()

            running_total += loss.item()
            running_data += loss_data.item()
            running_colloc += loss_colloc.item()
            running_colloc_used += n_used

            if (ib % PRINT_BATCH_EVERY) == 0:
                print(
                    f"[epoch {epoch:03d} | batch {ib:04d}/{len(loader):04d}] "
                    f"total={loss.item():.6e} "
                    f"data={loss_data.item():.6e} "
                    f"colloc={loss_colloc.item():.6e} "
                    f"used={n_used}"
                )

        epoch_total = running_total / max(len(loader), 1)
        epoch_data = running_data / max(len(loader), 1)
        epoch_colloc = running_colloc / max(len(loader), 1)
        loss_history.append([epoch, epoch_total, epoch_data, epoch_colloc])

        dt_epoch = time.time() - t0
        print(
            f"Epoch {epoch:03d} done | "
            f"total={epoch_total:.6e} | "
            f"data={epoch_data:.6e} | "
            f"colloc={epoch_colloc:.6e} | "
            f"time={dt_epoch:.1f}s | "
            f"colloc_used={running_colloc_used}"
        )

        last_ckpt = os.path.join(CKPT_DIR, "last_model.pt")
        save_checkpoint(last_ckpt, model, optimizer, epoch, loss_history, data_dir)

        if ((epoch + 1) % SAVE_EVERY_EPOCH) == 0:
            epoch_ckpt = os.path.join(CKPT_DIR, f"model_epoch_{epoch:03d}.pt")
            save_checkpoint(epoch_ckpt, model, optimizer, epoch, loss_history, data_dir)

        save_loss_csv(os.path.join(CKPT_DIR, "loss_history.csv"), loss_history)

    final_ckpt = os.path.join(CKPT_DIR, "final_model.pt")
    save_checkpoint(final_ckpt, model, optimizer, EPOCHS - 1, loss_history, data_dir)
    save_loss_csv(os.path.join(CKPT_DIR, "loss_history.csv"), loss_history)

    print("[Train] finished.")
    print(f"[Train] final checkpoint: {final_ckpt}")
    return model, loss_history

# ------------------------------------------------------------
# Main
# ------------------------------------------------------------
if __name__ == "__main__":
    train()

Using device: cuda
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
[AutoDetect] checking candidate snapshot roots...
  /content/drive/MyDrive/AI_4DVAR/klein_ckpt_1L
     valid IC dirs = 28, total pairs = 168
  /content/drive/MyDrive/AI_4DVAR/klein_ckpt_AL_centers_colloc
     valid IC dirs = 0, total pairs = 0
  /content/drive/MyDrive/AI_4DVAR/klein_ckpt_AL_centers
     valid IC dirs = 0, total pairs = 0
  /content/drive/MyDrive/AI_4DVAR/klein_ckpt_1L_colloc
     valid IC dirs = 0, total pairs = 0
[AutoDetect] using: /content/drive/MyDrive/AI_4DVAR/klein_ckpt_1L
[CollocBank] loaded gauss_00 | samples=15616 | macro_groups=7
[CollocBank] loaded gauss_01 | samples=15616 | macro_groups=7
[CollocBank] loaded gauss_02 | samples=15616 | macro_groups=7
[CollocBank] loaded gauss_03 | samples=15616 | macro_groups=7
[CollocBank] loaded gauss_04 | samples=15616 | macro_groups=7
[CollocBank] loaded gauss_05 | samples=15616 | macro_groups=7
[CollocBank] loaded mix_00 | samples=15616 | macro_groups=

# Next step in training

In [ ]:
# ============================================================
# train_gnn_1layer_tendency_colloc_wave.py
# ------------------------------------------------------------
# Hybrid prototype:
#   - tendency-head emulator
#   - collocation tendency matching loss
#   - continuity-like wave residual loss
#
# Loss:
#   L = L_data
#     + lambda_colloc * L_colloc_tend
#     + lambda_wave   * L_wave_cont
#
# where:
#   L_wave_cont = || eta_t_pred + H * div_fd ||^2
#
# This is the first wave-physics extension.
# ============================================================

import os
import sys
import glob
import csv
import time
import random
import numpy as np
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

# ------------------------------------------------------------
# Import colloc_bank
# ------------------------------------------------------------
sys.path.append("/content/drive/MyDrive/AI_4DVAR")
from colloc_bank import CollocBank

# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ------------------------------------------------------------
# Device
# ------------------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------
ROOT = "/content/drive/MyDrive/AI_4DVAR"

DATA_DIR_CANDIDATES = [
    f"{ROOT}/klein_ckpt_1L",
    f"{ROOT}/klein_ckpt_AL_centers_colloc",
    f"{ROOT}/klein_ckpt_AL_centers",
    f"{ROOT}/klein_ckpt_1L_colloc",
]

COLLOC_DIR = f"{ROOT}/klein_ml_1L/colloc_raw"
CKPT_DIR = f"{ROOT}/klein_ml_1L_ckpt_tendency_colloc_wave"
os.makedirs(CKPT_DIR, exist_ok=True)

# ------------------------------------------------------------
# Config
# ------------------------------------------------------------
BATCH_SIZE = 2
LR = 1e-3
EPOCHS = 20
WEIGHT_DECAY = 1e-6
GRAD_CLIP = 1.0

ROLL_STEPS = 2
ROLL_GAMMA = 0.9
DT_MACRO = 200.0 * 30.0  # 6000 s

MAX_PAIRS_PER_IC = 6
NUM_WORKERS = 0
PIN_MEMORY = torch.cuda.is_available()

USE_COLLOC = True
N_COLLOC_BATCH = 64

# based on your previous runs
LAMBDA_COLLOC = 100.0

# new wave residual term
LAMBDA_WAVE = 100.0

SAVE_EVERY_EPOCH = 5
RESUME_FROM = None
PRINT_BATCH_EVERY = 20

# ------------------------------------------------------------
# Helper: detect valid data root
# ------------------------------------------------------------
def count_pairs_in_root(data_dir):
    if not os.path.exists(data_dir):
        return 0, 0

    ic_dirs = [d for d in glob.glob(os.path.join(data_dir, "*")) if os.path.isdir(d)]
    n_ic = 0
    n_pairs = 0

    for ic_dir in ic_dirs:
        files = sorted(glob.glob(os.path.join(ic_dir, "klein_step_*.npz")))
        if len(files) >= 2:
            n_ic += 1
            n_pairs += (len(files) - 1)

    return n_ic, n_pairs

def autodetect_data_dir(candidates):
    print("[AutoDetect] checking candidate snapshot roots...")
    best_dir = None
    best_pairs = -1

    for d in candidates:
        n_ic, n_pairs = count_pairs_in_root(d)
        print(f"  {d}")
        print(f"     valid IC dirs = {n_ic}, total pairs = {n_pairs}")
        if n_pairs > best_pairs:
            best_pairs = n_pairs
            best_dir = d

    if best_dir is None or best_pairs <= 0:
        msg = "No valid snapshot root found.\n"
        msg += "Checked these paths:\n"
        for d in candidates:
            msg += f"  - {d}\n"
        msg += "Expected structure:\n"
        msg += "  ROOT/<IC_KEY>/klein_step_XXXXXX.npz\n"
        raise RuntimeError(msg)

    print(f"[AutoDetect] using: {best_dir}")
    return best_dir

# ------------------------------------------------------------
# Dataset
# ------------------------------------------------------------
class KleinDataset(Dataset):
    """
    Consecutive snapshot pairs.

    macro_index is aligned with collocation generation:
      pair i -> macro_index = i + 1
    """
    def __init__(self, data_dir, max_pairs_per_ic=None):
        self.samples = []

        print(f"[Dataset] scanning data_dir: {data_dir}")
        if not os.path.exists(data_dir):
            raise RuntimeError(f"DATA_DIR does not exist: {data_dir}")

        ic_dirs = sorted([d for d in glob.glob(os.path.join(data_dir, "*")) if os.path.isdir(d)])
        print(f"[Dataset] found {len(ic_dirs)} subdirectories")

        for ic_dir in ic_dirs:
            ic_key = os.path.basename(ic_dir)
            files = sorted(glob.glob(os.path.join(ic_dir, "klein_step_*.npz")))
            print(f"[Dataset] {ic_key:20s} -> {len(files)} snapshot files")

            if len(files) < 2:
                continue

            pairs = []
            for i in range(len(files) - 1):
                f0 = files[i]
                f1 = files[i + 1]
                macro_index = i + 1
                pairs.append((f0, f1, ic_key, macro_index))

            if max_pairs_per_ic is not None:
                pairs = pairs[:max_pairs_per_ic]

            self.samples.extend(pairs)

        if len(self.samples) == 0:
            raise RuntimeError(
                f"No training pairs found in {data_dir}.\n"
                f"Expected files like: <IC_KEY>/klein_step_XXXXXX.npz"
            )

        print(f"[Dataset] total training pairs = {len(self.samples)}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        f0, f1, ic_key, macro_index = self.samples[idx]

        z0 = np.load(f0)
        z1 = np.load(f1)

        x = np.stack([z0["eta"], z0["uc"], z0["vc"]], axis=0).astype(np.float32)
        y = np.stack([z1["eta"], z1["uc"], z1["vc"]], axis=0).astype(np.float32)

        return {
            "x": torch.from_numpy(x),
            "y": torch.from_numpy(y),
            "ic_key": ic_key,
            "macro_index": np.int32(macro_index),
            "f0": f0,
            "f1": f1,
        }

# ------------------------------------------------------------
# Model
# ------------------------------------------------------------
class SimpleCNN(nn.Module):
    """
    Predicts tendencies:
        input  = [eta, uc, vc]
        output = [eta_t, uc_t, vc_t]
    """
    def __init__(self, in_ch=3, hidden=64, out_ch=3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, hidden, kernel_size=3, padding=1),
            nn.GELU(),
            nn.Conv2d(hidden, hidden, kernel_size=3, padding=1),
            nn.GELU(),
            nn.Conv2d(hidden, hidden, kernel_size=3, padding=1),
            nn.GELU(),
            nn.Conv2d(hidden, out_ch, kernel_size=3, padding=1),
        )

    def forward(self, x):
        return self.net(x)

# ------------------------------------------------------------
# Loss helpers
# ------------------------------------------------------------
def one_step_update(x, xdot, dt_macro):
    return x + dt_macro * xdot

def rollout_loss(model, x0, y_true, dt_macro, roll_steps=2, gamma=0.9):
    x = x0
    total = 0.0

    for k in range(roll_steps):
        xdot = model(x)
        x = one_step_update(x, xdot, dt_macro)
        loss_k = ((x - y_true) ** 2).mean()
        total = total + (gamma ** k) * loss_k

    return total

def colloc_losses_for_batch(xdot_pred, batch_x, batch_meta, colloc_bank, n_colloc_batch, H_value):
    """
    Compute two collocation-based losses:

      1. tendency matching:
         ||eta_t_pred - eta_t_fd||^2 + ||uc_t_pred - uc_t_fd||^2 + ||vc_t_pred - vc_t_fd||^2

      2. continuity-like wave residual:
         || eta_t_pred + H * div_fd ||^2

    Parameters
    ----------
    xdot_pred : [B, 3, ny, nx]
    batch_x   : [B, 3, ny, nx]  current state (not used yet, but kept for easy extension)
    batch_meta: dict with ic_key, macro_index
    """
    B = xdot_pred.shape[0]

    total_colloc = 0.0
    total_wave = 0.0
    n_used = 0

    ic_keys = batch_meta["ic_key"]
    macro_indices = batch_meta["macro_index"]

    for b in range(B):
        ic_key = ic_keys[b]
        macro_index = int(macro_indices[b])

        if not colloc_bank.has_ic(ic_key):
            continue

        colloc = colloc_bank.sample_nearest_macro(
            ic_key=ic_key,
            macro_index=macro_index,
            npts=n_colloc_batch,
            replace=False,
        )

        iy = torch.as_tensor(colloc["iy"], dtype=torch.long, device=xdot_pred.device)
        ix = torch.as_tensor(colloc["ix"], dtype=torch.long, device=xdot_pred.device)

        pred_eta_t = xdot_pred[b, 0, iy, ix]
        pred_uc_t  = xdot_pred[b, 1, iy, ix]
        pred_vc_t  = xdot_pred[b, 2, iy, ix]

        true_eta_t = torch.as_tensor(colloc["deta_dt_fd"], dtype=torch.float32, device=xdot_pred.device)
        true_uc_t  = torch.as_tensor(colloc["duc_dt_fd"],  dtype=torch.float32, device=xdot_pred.device)
        true_vc_t  = torch.as_tensor(colloc["dvc_dt_fd"],  dtype=torch.float32, device=xdot_pred.device)

        div_fd = torch.as_tensor(colloc["div_fd"], dtype=torch.float32, device=xdot_pred.device)

        # tendency matching
        loss_colloc_b = ((pred_eta_t - true_eta_t) ** 2).mean()
        loss_colloc_b = loss_colloc_b + ((pred_uc_t - true_uc_t) ** 2).mean()
        loss_colloc_b = loss_colloc_b + ((pred_vc_t - true_vc_t) ** 2).mean()

        # continuity-like wave residual
        loss_wave_b = ((pred_eta_t + H_value * div_fd) ** 2).mean()

        total_colloc = total_colloc + loss_colloc_b
        total_wave = total_wave + loss_wave_b
        n_used += 1

    if n_used == 0:
        zero = torch.tensor(0.0, device=xdot_pred.device)
        return zero, zero, 0

    return total_colloc / n_used, total_wave / n_used, n_used

# ------------------------------------------------------------
# Checkpoint helpers
# ------------------------------------------------------------
def save_checkpoint(path, model, optimizer, epoch, loss_history, data_dir):
    torch.save(
        {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "loss_history": loss_history,
            "data_dir": data_dir,
        },
        path,
    )

def load_checkpoint(path, model, optimizer=None):
    ckpt = torch.load(path, map_location=device)
    model.load_state_dict(ckpt["model_state_dict"])
    if optimizer is not None and "optimizer_state_dict" in ckpt:
        optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    epoch = ckpt.get("epoch", -1)
    loss_history = ckpt.get("loss_history", [])
    return epoch, loss_history, ckpt

# ------------------------------------------------------------
# Logging
# ------------------------------------------------------------
def save_loss_csv(path, loss_history):
    with open(path, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["epoch", "train_total", "train_data", "train_colloc", "train_wave"])
        for row in loss_history:
            writer.writerow(row)

# ------------------------------------------------------------
# Train
# ------------------------------------------------------------
def train():
    data_dir = autodetect_data_dir(DATA_DIR_CANDIDATES)

    if not os.path.exists(COLLOC_DIR):
        raise RuntimeError(f"COLLOC_DIR does not exist: {COLLOC_DIR}")

    colloc_bank = CollocBank(COLLOC_DIR, verbose=True)

    dataset = KleinDataset(data_dir, max_pairs_per_ic=MAX_PAIRS_PER_IC)
    loader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        drop_last=False,
    )

    model = SimpleCNN().to(device)
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=LR,
        weight_decay=WEIGHT_DECAY,
    )

    start_epoch = 0
    loss_history = []

    if RESUME_FROM is not None and os.path.exists(RESUME_FROM):
        print(f"[Resume] loading checkpoint: {RESUME_FROM}")
        loaded_epoch, loaded_history, ckpt = load_checkpoint(RESUME_FROM, model, optimizer)
        start_epoch = loaded_epoch + 1
        loss_history = loaded_history
        print(f"[Resume] starting from epoch {start_epoch}")
        print(f"[Resume] checkpoint data_dir was: {ckpt.get('data_dir', 'unknown')}")

    print(f"[Train] using data_dir   = {data_dir}")
    print(f"[Train] using colloc_dir = {COLLOC_DIR}")
    print(f"[Train] dataset pairs    = {len(dataset)}")
    print(f"[Train] batches/epoch    = {len(loader)}")
    print(f"[Train] LAMBDA_COLLOC    = {LAMBDA_COLLOC}")
    print(f"[Train] LAMBDA_WAVE      = {LAMBDA_WAVE}")

    for epoch in range(start_epoch, EPOCHS):
        t0 = time.time()
        model.train()

        running_total = 0.0
        running_data = 0.0
        running_colloc = 0.0
        running_wave = 0.0
        running_colloc_used = 0

        for ib, batch in enumerate(loader):
            x = batch["x"].to(device, non_blocking=True)
            y = batch["y"].to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            # data loss via tendency rollout
            loss_data = rollout_loss(
                model=model,
                x0=x,
                y_true=y,
                dt_macro=DT_MACRO,
                roll_steps=ROLL_STEPS,
                gamma=ROLL_GAMMA,
            )

            # first-step tendency field for collocation losses
            xdot_pred = model(x)

            if USE_COLLOC:
                loss_colloc, loss_wave, n_used = colloc_losses_for_batch(
                    xdot_pred=xdot_pred,
                    batch_x=x,
                    batch_meta=batch,
                    colloc_bank=colloc_bank,
                    n_colloc_batch=N_COLLOC_BATCH,
                    H_value=1000.0,  # or use metadata later if needed
                )
            else:
                loss_colloc = torch.tensor(0.0, device=device)
                loss_wave = torch.tensor(0.0, device=device)
                n_used = 0

            loss = (
                loss_data
                + LAMBDA_COLLOC * loss_colloc
                + LAMBDA_WAVE * loss_wave
            )

            if not torch.isfinite(loss):
                raise RuntimeError(f"Non-finite loss at epoch={epoch}, batch={ib}: {loss.item()}")

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step()

            running_total += loss.item()
            running_data += loss_data.item()
            running_colloc += loss_colloc.item()
            running_wave += loss_wave.item()
            running_colloc_used += n_used

            if (ib % PRINT_BATCH_EVERY) == 0:
                print(
                    f"[epoch {epoch:03d} | batch {ib:04d}/{len(loader):04d}] "
                    f"total={loss.item():.6e} "
                    f"data={loss_data.item():.6e} "
                    f"colloc={loss_colloc.item():.6e} "
                    f"wave={loss_wave.item():.6e} "
                    f"used={n_used}"
                )

        epoch_total = running_total / max(len(loader), 1)
        epoch_data = running_data / max(len(loader), 1)
        epoch_colloc = running_colloc / max(len(loader), 1)
        epoch_wave = running_wave / max(len(loader), 1)

        loss_history.append([epoch, epoch_total, epoch_data, epoch_colloc, epoch_wave])

        dt_epoch = time.time() - t0
        print(
            f"Epoch {epoch:03d} done | "
            f"total={epoch_total:.6e} | "
            f"data={epoch_data:.6e} | "
            f"colloc={epoch_colloc:.6e} | "
            f"wave={epoch_wave:.6e} | "
            f"time={dt_epoch:.1f}s | "
            f"colloc_used={running_colloc_used}"
        )

        last_ckpt = os.path.join(CKPT_DIR, "last_model.pt")
        save_checkpoint(last_ckpt, model, optimizer, epoch, loss_history, data_dir)

        if ((epoch + 1) % SAVE_EVERY_EPOCH) == 0:
            epoch_ckpt = os.path.join(CKPT_DIR, f"model_epoch_{epoch:03d}.pt")
            save_checkpoint(epoch_ckpt, model, optimizer, epoch, loss_history, data_dir)

        save_loss_csv(os.path.join(CKPT_DIR, "loss_history.csv"), loss_history)

    final_ckpt = os.path.join(CKPT_DIR, "final_model.pt")
    save_checkpoint(final_ckpt, model, optimizer, EPOCHS - 1, loss_history, data_dir)
    save_loss_csv(os.path.join(CKPT_DIR, "loss_history.csv"), loss_history)

    print("[Train] finished.")
    print(f"[Train] final checkpoint: {final_ckpt}")
    return model, loss_history

# ------------------------------------------------------------
# Main
# ------------------------------------------------------------
if __name__ == "__main__":
    train()

Using device: cuda
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
[AutoDetect] checking candidate snapshot roots...
  /content/drive/MyDrive/AI_4DVAR/klein_ckpt_1L
     valid IC dirs = 28, total pairs = 168
  /content/drive/MyDrive/AI_4DVAR/klein_ckpt_AL_centers_colloc
     valid IC dirs = 0, total pairs = 0
  /content/drive/MyDrive/AI_4DVAR/klein_ckpt_AL_centers
     valid IC dirs = 0, total pairs = 0
  /content/drive/MyDrive/AI_4DVAR/klein_ckpt_1L_colloc
     valid IC dirs = 0, total pairs = 0
[AutoDetect] using: /content/drive/MyDrive/AI_4DVAR/klein_ckpt_1L
[CollocBank] loaded gauss_00 | samples=15616 | macro_groups=7
[CollocBank] loaded gauss_01 | samples=15616 | macro_groups=7
[CollocBank] loaded gauss_02 | samples=15616 | macro_groups=7
[CollocBank] loaded gauss_03 | samples=15616 | macro_groups=7
[CollocBank] loaded gauss_04 | samples=15616 | macro_groups=7
[CollocBank] loaded gauss_05 | samples=15616 | macro_groups=7
[CollocBank] loaded mix_00 | samples=15616 | macro_groups=

# Adding more constraints

In [ ]:
# ============================================================
# train_gnn_1layer_tendency_colloc_wave_mom.py
# ------------------------------------------------------------
# Hybrid prototype:
#   - tendency-head emulator
#   - collocation tendency matching loss
#   - continuity-like wave residual
#   - momentum-like wave residual
#
# Loss:
#   L = L_data
#     + lambda_colloc * L_colloc_tend
#     + lambda_wave   * L_wave_cont
#     + lambda_mom    * L_wave_mom
#
# where:
#   L_wave_cont = || eta_t_pred + H * div_fd ||^2
#   L_wave_mom  = || u_t_pred - f*v + g*etax_fd ||^2
#               + || v_t_pred + f*u + g*etay_fd ||^2
#
# This is the first full wave-aware hybrid trainer.
# ============================================================

import os
import sys
import glob
import csv
import time
import random
import numpy as np
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

# ------------------------------------------------------------
# Import colloc_bank
# ------------------------------------------------------------
sys.path.append("/content/drive/MyDrive/AI_4DVAR")
from colloc_bank import CollocBank

# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ------------------------------------------------------------
# Device
# ------------------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------
ROOT = "/content/drive/MyDrive/AI_4DVAR"

DATA_DIR_CANDIDATES = [
    f"{ROOT}/klein_ckpt_1L",
    f"{ROOT}/klein_ckpt_AL_centers_colloc",
    f"{ROOT}/klein_ckpt_AL_centers",
    f"{ROOT}/klein_ckpt_1L_colloc",
]

COLLOC_DIR = f"{ROOT}/klein_ml_1L/colloc_raw"
CKPT_DIR = f"{ROOT}/klein_ml_1L_ckpt_tendency_colloc_wave_mom"
os.makedirs(CKPT_DIR, exist_ok=True)

# ------------------------------------------------------------
# Config
# ------------------------------------------------------------
BATCH_SIZE = 2
LR = 1e-3
EPOCHS = 20
WEIGHT_DECAY = 1e-6
GRAD_CLIP = 1.0

ROLL_STEPS = 2
ROLL_GAMMA = 0.9
DT_MACRO = 200.0 * 30.0  # 6000 s

MAX_PAIRS_PER_IC = 6
NUM_WORKERS = 0
PIN_MEMORY = torch.cuda.is_available()

USE_COLLOC = True
N_COLLOC_BATCH = 64

LAMBDA_COLLOC = 100.0
LAMBDA_WAVE   = 100.0
LAMBDA_MOM    = 100.0

# Physical constants for residuals
H_VALUE = 1000.0
G_VALUE = 9.81

SAVE_EVERY_EPOCH = 5
RESUME_FROM = None
PRINT_BATCH_EVERY = 20

# ------------------------------------------------------------
# Helper: detect valid data root
# ------------------------------------------------------------
def count_pairs_in_root(data_dir):
    if not os.path.exists(data_dir):
        return 0, 0

    ic_dirs = [d for d in glob.glob(os.path.join(data_dir, "*")) if os.path.isdir(d)]
    n_ic = 0
    n_pairs = 0

    for ic_dir in ic_dirs:
        files = sorted(glob.glob(os.path.join(ic_dir, "klein_step_*.npz")))
        if len(files) >= 2:
            n_ic += 1
            n_pairs += (len(files) - 1)

    return n_ic, n_pairs

def autodetect_data_dir(candidates):
    print("[AutoDetect] checking candidate snapshot roots...")
    best_dir = None
    best_pairs = -1

    for d in candidates:
        n_ic, n_pairs = count_pairs_in_root(d)
        print(f"  {d}")
        print(f"     valid IC dirs = {n_ic}, total pairs = {n_pairs}")
        if n_pairs > best_pairs:
            best_pairs = n_pairs
            best_dir = d

    if best_dir is None or best_pairs <= 0:
        msg = "No valid snapshot root found.\n"
        msg += "Checked these paths:\n"
        for d in candidates:
            msg += f"  - {d}\n"
        msg += "Expected structure:\n"
        msg += "  ROOT/<IC_KEY>/klein_step_XXXXXX.npz\n"
        raise RuntimeError(msg)

    print(f"[AutoDetect] using: {best_dir}")
    return best_dir

# ------------------------------------------------------------
# Dataset
# ------------------------------------------------------------
class KleinDataset(Dataset):
    """
    Consecutive snapshot pairs.

    macro_index aligned with collocation generation:
      pair i -> macro_index = i + 1
    """
    def __init__(self, data_dir, max_pairs_per_ic=None):
        self.samples = []

        print(f"[Dataset] scanning data_dir: {data_dir}")
        if not os.path.exists(data_dir):
            raise RuntimeError(f"DATA_DIR does not exist: {data_dir}")

        ic_dirs = sorted([d for d in glob.glob(os.path.join(data_dir, "*")) if os.path.isdir(d)])
        print(f"[Dataset] found {len(ic_dirs)} subdirectories")

        for ic_dir in ic_dirs:
            ic_key = os.path.basename(ic_dir)
            files = sorted(glob.glob(os.path.join(ic_dir, "klein_step_*.npz")))
            print(f"[Dataset] {ic_key:20s} -> {len(files)} snapshot files")

            if len(files) < 2:
                continue

            pairs = []
            for i in range(len(files) - 1):
                f0 = files[i]
                f1 = files[i + 1]
                macro_index = i + 1
                pairs.append((f0, f1, ic_key, macro_index))

            if max_pairs_per_ic is not None:
                pairs = pairs[:max_pairs_per_ic]

            self.samples.extend(pairs)

        if len(self.samples) == 0:
            raise RuntimeError(
                f"No training pairs found in {data_dir}.\n"
                f"Expected files like: <IC_KEY>/klein_step_XXXXXX.npz"
            )

        print(f"[Dataset] total training pairs = {len(self.samples)}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        f0, f1, ic_key, macro_index = self.samples[idx]

        z0 = np.load(f0)
        z1 = np.load(f1)

        x = np.stack([z0["eta"], z0["uc"], z0["vc"]], axis=0).astype(np.float32)
        y = np.stack([z1["eta"], z1["uc"], z1["vc"]], axis=0).astype(np.float32)

        return {
            "x": torch.from_numpy(x),
            "y": torch.from_numpy(y),
            "ic_key": ic_key,
            "macro_index": np.int32(macro_index),
            "f0": f0,
            "f1": f1,
        }

# ------------------------------------------------------------
# Model
# ------------------------------------------------------------
class SimpleCNN(nn.Module):
    """
    Predicts tendencies:
        input  = [eta, uc, vc]
        output = [eta_t, uc_t, vc_t]
    """
    def __init__(self, in_ch=3, hidden=64, out_ch=3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, hidden, kernel_size=3, padding=1),
            nn.GELU(),
            nn.Conv2d(hidden, hidden, kernel_size=3, padding=1),
            nn.GELU(),
            nn.Conv2d(hidden, hidden, kernel_size=3, padding=1),
            nn.GELU(),
            nn.Conv2d(hidden, out_ch, kernel_size=3, padding=1),
        )

    def forward(self, x):
        return self.net(x)

# ------------------------------------------------------------
# Loss helpers
# ------------------------------------------------------------
def one_step_update(x, xdot, dt_macro):
    return x + dt_macro * xdot

def rollout_loss(model, x0, y_true, dt_macro, roll_steps=2, gamma=0.9):
    x = x0
    total = 0.0

    for k in range(roll_steps):
        xdot = model(x)
        x = one_step_update(x, xdot, dt_macro)
        loss_k = ((x - y_true) ** 2).mean()
        total = total + (gamma ** k) * loss_k

    return total

def colloc_losses_for_batch(xdot_pred, batch_x, batch_meta, colloc_bank, n_colloc_batch, H_value, G_value):
    """
    Compute three collocation-based losses:

      1. tendency matching:
         ||eta_t_pred - eta_t_fd||^2 + ||uc_t_pred - uc_t_fd||^2 + ||vc_t_pred - vc_t_fd||^2

      2. continuity-like wave residual:
         || eta_t_pred + H * div_fd ||^2

      3. momentum-like wave residual:
         || uc_t_pred - f*vc + g*etax_fd ||^2
       + || vc_t_pred + f*uc + g*etay_fd ||^2

    batch_x : [B, 3, ny, nx] current state = [eta, uc, vc]
    """
    B = xdot_pred.shape[0]

    total_colloc = 0.0
    total_wave = 0.0
    total_mom = 0.0
    n_used = 0

    ic_keys = batch_meta["ic_key"]
    macro_indices = batch_meta["macro_index"]

    for b in range(B):
        ic_key = ic_keys[b]
        macro_index = int(macro_indices[b])

        if not colloc_bank.has_ic(ic_key):
            continue

        colloc = colloc_bank.sample_nearest_macro(
            ic_key=ic_key,
            macro_index=macro_index,
            npts=n_colloc_batch,
            replace=False,
        )

        iy = torch.as_tensor(colloc["iy"], dtype=torch.long, device=xdot_pred.device)
        ix = torch.as_tensor(colloc["ix"], dtype=torch.long, device=xdot_pred.device)

        # Model-predicted tendencies at collocation points
        pred_eta_t = xdot_pred[b, 0, iy, ix]
        pred_uc_t  = xdot_pred[b, 1, iy, ix]
        pred_vc_t  = xdot_pred[b, 2, iy, ix]

        # Current state at the same collocation points
        state_eta = batch_x[b, 0, iy, ix]
        state_uc  = batch_x[b, 1, iy, ix]
        state_vc  = batch_x[b, 2, iy, ix]

        # Saved FD tendency targets
        true_eta_t = torch.as_tensor(colloc["deta_dt_fd"], dtype=torch.float32, device=xdot_pred.device)
        true_uc_t  = torch.as_tensor(colloc["duc_dt_fd"],  dtype=torch.float32, device=xdot_pred.device)
        true_vc_t  = torch.as_tensor(colloc["dvc_dt_fd"],  dtype=torch.float32, device=xdot_pred.device)

        # Saved FD local diagnostics
        div_fd  = torch.as_tensor(colloc["div_fd"],  dtype=torch.float32, device=xdot_pred.device)
        etax_fd = torch.as_tensor(colloc["etax_fd"], dtype=torch.float32, device=xdot_pred.device)
        etay_fd = torch.as_tensor(colloc["etay_fd"], dtype=torch.float32, device=xdot_pred.device)
        f_fd    = torch.as_tensor(colloc["f"],       dtype=torch.float32, device=xdot_pred.device)

        # 1) Tendency matching
        loss_colloc_b = ((pred_eta_t - true_eta_t) ** 2).mean()
        loss_colloc_b = loss_colloc_b + ((pred_uc_t - true_uc_t) ** 2).mean()
        loss_colloc_b = loss_colloc_b + ((pred_vc_t - true_vc_t) ** 2).mean()

        # 2) Continuity-like residual
        loss_wave_b = ((pred_eta_t + H_value * div_fd) ** 2).mean()

        # 3) Momentum-like residual
        #    u_t - f v + g eta_x = 0
        #    v_t + f u + g eta_y = 0
        Ru = pred_uc_t - f_fd * state_vc + G_value * etax_fd
        Rv = pred_vc_t + f_fd * state_uc + G_value * etay_fd
        loss_mom_b = (Ru ** 2).mean() + (Rv ** 2).mean()

        total_colloc = total_colloc + loss_colloc_b
        total_wave = total_wave + loss_wave_b
        total_mom = total_mom + loss_mom_b
        n_used += 1

    if n_used == 0:
        zero = torch.tensor(0.0, device=xdot_pred.device)
        return zero, zero, zero, 0

    return total_colloc / n_used, total_wave / n_used, total_mom / n_used, n_used

# ------------------------------------------------------------
# Checkpoint helpers
# ------------------------------------------------------------
def save_checkpoint(path, model, optimizer, epoch, loss_history, data_dir):
    torch.save(
        {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "loss_history": loss_history,
            "data_dir": data_dir,
        },
        path,
    )

def load_checkpoint(path, model, optimizer=None):
    ckpt = torch.load(path, map_location=device)
    model.load_state_dict(ckpt["model_state_dict"])
    if optimizer is not None and "optimizer_state_dict" in ckpt:
        optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    epoch = ckpt.get("epoch", -1)
    loss_history = ckpt.get("loss_history", [])
    return epoch, loss_history, ckpt

# ------------------------------------------------------------
# Logging
# ------------------------------------------------------------
def save_loss_csv(path, loss_history):
    with open(path, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["epoch", "train_total", "train_data", "train_colloc", "train_wave", "train_mom"])
        for row in loss_history:
            writer.writerow(row)

# ------------------------------------------------------------
# Train
# ------------------------------------------------------------
def train():
    data_dir = autodetect_data_dir(DATA_DIR_CANDIDATES)

    if not os.path.exists(COLLOC_DIR):
        raise RuntimeError(f"COLLOC_DIR does not exist: {COLLOC_DIR}")

    colloc_bank = CollocBank(COLLOC_DIR, verbose=True)

    dataset = KleinDataset(data_dir, max_pairs_per_ic=MAX_PAIRS_PER_IC)
    loader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        drop_last=False,
    )

    model = SimpleCNN().to(device)
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=LR,
        weight_decay=WEIGHT_DECAY,
    )

    start_epoch = 0
    loss_history = []

    if RESUME_FROM is not None and os.path.exists(RESUME_FROM):
        print(f"[Resume] loading checkpoint: {RESUME_FROM}")
        loaded_epoch, loaded_history, ckpt = load_checkpoint(RESUME_FROM, model, optimizer)
        start_epoch = loaded_epoch + 1
        loss_history = loaded_history
        print(f"[Resume] starting from epoch {start_epoch}")
        print(f"[Resume] checkpoint data_dir was: {ckpt.get('data_dir', 'unknown')}")

    print(f"[Train] using data_dir   = {data_dir}")
    print(f"[Train] using colloc_dir = {COLLOC_DIR}")
    print(f"[Train] dataset pairs    = {len(dataset)}")
    print(f"[Train] batches/epoch    = {len(loader)}")
    print(f"[Train] LAMBDA_COLLOC    = {LAMBDA_COLLOC}")
    print(f"[Train] LAMBDA_WAVE      = {LAMBDA_WAVE}")
    print(f"[Train] LAMBDA_MOM       = {LAMBDA_MOM}")

    for epoch in range(start_epoch, EPOCHS):
        t0 = time.time()
        model.train()

        running_total = 0.0
        running_data = 0.0
        running_colloc = 0.0
        running_wave = 0.0
        running_mom = 0.0
        running_colloc_used = 0

        for ib, batch in enumerate(loader):
            x = batch["x"].to(device, non_blocking=True)
            y = batch["y"].to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            # data loss via tendency rollout
            loss_data = rollout_loss(
                model=model,
                x0=x,
                y_true=y,
                dt_macro=DT_MACRO,
                roll_steps=ROLL_STEPS,
                gamma=ROLL_GAMMA,
            )

            # first-step tendency field for collocation losses
            xdot_pred = model(x)

            if USE_COLLOC:
                loss_colloc, loss_wave, loss_mom, n_used = colloc_losses_for_batch(
                    xdot_pred=xdot_pred,
                    batch_x=x,
                    batch_meta=batch,
                    colloc_bank=colloc_bank,
                    n_colloc_batch=N_COLLOC_BATCH,
                    H_value=H_VALUE,
                    G_value=G_VALUE,
                )
            else:
                loss_colloc = torch.tensor(0.0, device=device)
                loss_wave = torch.tensor(0.0, device=device)
                loss_mom = torch.tensor(0.0, device=device)
                n_used = 0

            loss = (
                loss_data
                + LAMBDA_COLLOC * loss_colloc
                + LAMBDA_WAVE * loss_wave
                + LAMBDA_MOM * loss_mom
            )

            if not torch.isfinite(loss):
                raise RuntimeError(f"Non-finite loss at epoch={epoch}, batch={ib}: {loss.item()}")

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step()

            running_total += loss.item()
            running_data += loss_data.item()
            running_colloc += loss_colloc.item()
            running_wave += loss_wave.item()
            running_mom += loss_mom.item()
            running_colloc_used += n_used

            if (ib % PRINT_BATCH_EVERY) == 0:
                print(
                    f"[epoch {epoch:03d} | batch {ib:04d}/{len(loader):04d}] "
                    f"total={loss.item():.6e} "
                    f"data={loss_data.item():.6e} "
                    f"colloc={loss_colloc.item():.6e} "
                    f"wave={loss_wave.item():.6e} "
                    f"mom={loss_mom.item():.6e} "
                    f"used={n_used}"
                )

        epoch_total = running_total / max(len(loader), 1)
        epoch_data = running_data / max(len(loader), 1)
        epoch_colloc = running_colloc / max(len(loader), 1)
        epoch_wave = running_wave / max(len(loader), 1)
        epoch_mom = running_mom / max(len(loader), 1)

        loss_history.append([epoch, epoch_total, epoch_data, epoch_colloc, epoch_wave, epoch_mom])

        dt_epoch = time.time() - t0
        print(
            f"Epoch {epoch:03d} done | "
            f"total={epoch_total:.6e} | "
            f"data={epoch_data:.6e} | "
            f"colloc={epoch_colloc:.6e} | "
            f"wave={epoch_wave:.6e} | "
            f"mom={epoch_mom:.6e} | "
            f"time={dt_epoch:.1f}s | "
            f"colloc_used={running_colloc_used}"
        )

        last_ckpt = os.path.join(CKPT_DIR, "last_model.pt")
        save_checkpoint(last_ckpt, model, optimizer, epoch, loss_history, data_dir)

        if ((epoch + 1) % SAVE_EVERY_EPOCH) == 0:
            epoch_ckpt = os.path.join(CKPT_DIR, f"model_epoch_{epoch:03d}.pt")
            save_checkpoint(epoch_ckpt, model, optimizer, epoch, loss_history, data_dir)

        save_loss_csv(os.path.join(CKPT_DIR, "loss_history.csv"), loss_history)

    final_ckpt = os.path.join(CKPT_DIR, "final_model.pt")
    save_checkpoint(final_ckpt, model, optimizer, EPOCHS - 1, loss_history, data_dir)
    save_loss_csv(os.path.join(CKPT_DIR, "loss_history.csv"), loss_history)

    print("[Train] finished.")
    print(f"[Train] final checkpoint: {final_ckpt}")
    return model, loss_history

# ------------------------------------------------------------
# Main
# ------------------------------------------------------------
if __name__ == "__main__":
    train()

Using device: cuda
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
[AutoDetect] checking candidate snapshot roots...
  /content/drive/MyDrive/AI_4DVAR/klein_ckpt_1L
     valid IC dirs = 28, total pairs = 168
  /content/drive/MyDrive/AI_4DVAR/klein_ckpt_AL_centers_colloc
     valid IC dirs = 0, total pairs = 0
  /content/drive/MyDrive/AI_4DVAR/klein_ckpt_AL_centers
     valid IC dirs = 0, total pairs = 0
  /content/drive/MyDrive/AI_4DVAR/klein_ckpt_1L_colloc
     valid IC dirs = 0, total pairs = 0
[AutoDetect] using: /content/drive/MyDrive/AI_4DVAR/klein_ckpt_1L
[CollocBank] loaded gauss_00 | samples=15616 | macro_groups=7
[CollocBank] loaded gauss_01 | samples=15616 | macro_groups=7
[CollocBank] loaded gauss_02 | samples=15616 | macro_groups=7
[CollocBank] loaded gauss_03 | samples=15616 | macro_groups=7
[CollocBank] loaded gauss_04 | samples=15616 | macro_groups=7
[CollocBank] loaded gauss_05 | samples=15616 | macro_groups=7
[CollocBank] loaded mix_00 | samples=15616 | macro_groups=

# Evaluation

In [ ]:
# ============================================================
# eval_gnn_1layer_hybrid.py
# ------------------------------------------------------------
# Evaluate hybrid emulator:
#   - rollout vs FD truth
#   - RMSE vs time
#   - mass & energy diagnostics
#   - field plots
# ============================================================

import os
import glob
import numpy as np
import torch
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# Paths (EDIT if needed)
# ------------------------------------------------------------
ROOT = "/content/drive/MyDrive/AI_4DVAR"

DATA_DIR = f"{ROOT}/klein_ckpt_1L"
MODEL_PATH = f"{ROOT}/klein_ml_1L_ckpt_tendency_colloc_wave_mom/final_model.pt"
OUT_DIR = f"{ROOT}/eval_hybrid_1L"
os.makedirs(OUT_DIR, exist_ok=True)

# ------------------------------------------------------------
# Physics constants
# ------------------------------------------------------------
g = 9.81
H = 1000.0
dx = 2.0e7 / 256
dy = 8.0e6 / 128
DT_MACRO = 200 * 30.0  # 6000 s

# ------------------------------------------------------------
# Device
# ------------------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ------------------------------------------------------------
# Model (same as training)
# ------------------------------------------------------------
class SimpleCNN(torch.nn.Module):
    def __init__(self, in_ch=3, hidden=64, out_ch=3):
        super().__init__()
        self.net = torch.nn.Sequential(
            torch.nn.Conv2d(in_ch, hidden, 3, padding=1),
            torch.nn.GELU(),
            torch.nn.Conv2d(hidden, hidden, 3, padding=1),
            torch.nn.GELU(),
            torch.nn.Conv2d(hidden, hidden, 3, padding=1),
            torch.nn.GELU(),
            torch.nn.Conv2d(hidden, out_ch, 3, padding=1),
        )

    def forward(self, x):
        return self.net(x)

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------
def rmse(a, b):
    return np.sqrt(np.mean((a - b) ** 2))

def total_mass(eta):
    return np.sum((eta + H)) * dx * dy

def total_energy(eta, uc, vc):
    h = eta + H
    ke = 0.5 * h * (uc**2 + vc**2)
    pe = 0.5 * g * (eta**2)
    return np.sum((ke + pe) * dx * dy)

# ------------------------------------------------------------
# Load model
# ------------------------------------------------------------
model = SimpleCNN().to(device)
ckpt = torch.load(MODEL_PATH, map_location=device)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()

print("Loaded model:", MODEL_PATH)

# ------------------------------------------------------------
# List ICs
# ------------------------------------------------------------
ic_dirs = sorted([d for d in glob.glob(os.path.join(DATA_DIR, "*")) if os.path.isdir(d)])

# ------------------------------------------------------------
# Evaluate each IC
# ------------------------------------------------------------
for ic_dir in ic_dirs:

    ic_key = os.path.basename(ic_dir)
    print("\n=== IC:", ic_key, "===")

    files = sorted(glob.glob(os.path.join(ic_dir, "klein_step_*.npz")))

    if len(files) < 2:
        continue

    # Load truth trajectory
    truth = []
    for f in files:
        z = np.load(f)
        truth.append(np.stack([z["eta"], z["uc"], z["vc"]], axis=0))
    truth = np.stack(truth)  # [T, 3, ny, nx]

    T = truth.shape[0]

    # Initial state
    x = torch.tensor(truth[0:1], dtype=torch.float32, device=device)

    preds = []
    preds.append(x.cpu().numpy()[0])

    # Rollout
    with torch.no_grad():
        for t in range(1, T):
            xdot = model(x)
            x = x + DT_MACRO * xdot
            preds.append(x.cpu().numpy()[0])

    preds = np.stack(preds)

    # --------------------------------------------------------
    # Diagnostics
    # --------------------------------------------------------
    rmse_eta = []
    rmse_u = []
    rmse_v = []
    mass = []
    energy = []

    for t in range(T):
        eta_p, u_p, v_p = preds[t]
        eta_t, u_t, v_t = truth[t]

        rmse_eta.append(rmse(eta_p, eta_t))
        rmse_u.append(rmse(u_p, u_t))
        rmse_v.append(rmse(v_p, v_t))

        mass.append(total_mass(eta_p))
        energy.append(total_energy(eta_p, u_p, v_p))

    rmse_eta = np.array(rmse_eta)
    rmse_u = np.array(rmse_u)
    rmse_v = np.array(rmse_v)

    mass = np.array(mass)
    energy = np.array(energy)

    # --------------------------------------------------------
    # Save RMSE plot
    # --------------------------------------------------------
    plt.figure(figsize=(6,4))
    plt.plot(rmse_eta, label="eta")
    plt.plot(rmse_u, label="u")
    plt.plot(rmse_v, label="v")
    plt.title(f"RMSE vs time ({ic_key})")
    plt.xlabel("time step")
    plt.ylabel("RMSE")
    plt.legend()
    plt.grid(True)
    plt.savefig(f"{OUT_DIR}/{ic_key}_rmse.png", dpi=120)
    plt.close()

    # --------------------------------------------------------
    # Mass / energy
    # --------------------------------------------------------
    plt.figure(figsize=(6,4))
    plt.plot((mass - mass[0]) / mass[0], label="mass")
    plt.plot((energy - energy[0]) / energy[0], label="energy")
    plt.title(f"Mass/Energy drift ({ic_key})")
    plt.legend()
    plt.grid(True)
    plt.savefig(f"{OUT_DIR}/{ic_key}_mass_energy.png", dpi=120)
    plt.close()

    # --------------------------------------------------------
    # Field comparison (last step)
    # --------------------------------------------------------
    eta_p, u_p, v_p = preds[-1]
    eta_t, u_t, v_t = truth[-1]

    fig, axs = plt.subplots(2,3, figsize=(12,6))

    axs[0,0].imshow(eta_t, origin="lower"); axs[0,0].set_title("eta truth")
    axs[0,1].imshow(eta_p, origin="lower"); axs[0,1].set_title("eta pred")
    axs[0,2].imshow(eta_p - eta_t, origin="lower"); axs[0,2].set_title("eta error")

    axs[1,0].imshow(u_t, origin="lower"); axs[1,0].set_title("u truth")
    axs[1,1].imshow(u_p, origin="lower"); axs[1,1].set_title("u pred")
    axs[1,2].imshow(u_p - u_t, origin="lower"); axs[1,2].set_title("u error")

    plt.tight_layout()
    plt.savefig(f"{OUT_DIR}/{ic_key}_fields.png", dpi=120)
    plt.close()

    print("Saved results for:", ic_key)

print("\nEvaluation complete.")

Using device: cuda
Loaded model: /content/drive/MyDrive/AI_4DVAR/klein_ml_1L_ckpt_tendency_colloc_wave_mom/final_model.pt

=== IC: gauss_00 ===
Saved results for: gauss_00

=== IC: gauss_01 ===
Saved results for: gauss_01

=== IC: gauss_02 ===
Saved results for: gauss_02

=== IC: gauss_03 ===
Saved results for: gauss_03

=== IC: gauss_04 ===
Saved results for: gauss_04

=== IC: gauss_05 ===
Saved results for: gauss_05

=== IC: mix_00 ===
Saved results for: mix_00

=== IC: mix_01 ===
Saved results for: mix_01

=== IC: mix_02 ===
Saved results for: mix_02

=== IC: mix_03 ===
Saved results for: mix_03

=== IC: mix_04 ===
Saved results for: mix_04

=== IC: mix_05 ===
Saved results for: mix_05

=== IC: test_modon_00 ===
Saved results for: test_modon_00

=== IC: test_modon_01 ===
Saved results for: test_modon_01

=== IC: test_rh_00 ===
Saved results for: test_rh_00

=== IC: test_rh_01 ===
Saved results for: test_rh_01

=== IC: vort_00 ===
Saved results for: vort_00

=== IC: vort_01 ===
Saved

# Multistep physics

In [5]:
# ============================================================
# train_gnn_1layer_multistep_physics.py
# ------------------------------------------------------------
# Multi-step hybrid physics training
# ------------------------------------------------------------

import os, glob, random, time
import numpy as np
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

from colloc_bank import CollocBank

# ------------------------------------------------------------
# Config
# ------------------------------------------------------------
ROOT = "/content/drive/MyDrive/AI_4DVAR"

DATA_DIR = f"{ROOT}/klein_ckpt_1L"
COLLOC_DIR = f"{ROOT}/klein_ml_1L/colloc_raw"
CKPT_DIR = f"{ROOT}/klein_ml_1L_ckpt_multistep"

os.makedirs(CKPT_DIR, exist_ok=True)

BATCH_SIZE = 2
EPOCHS = 20
LR = 1e-3

DT = 6000.0
ROLL_STEPS = 4
GAMMA = 0.95

LAMBDA_COLLOC = 100.0
LAMBDA_WAVE = 100.0
LAMBDA_MOM = 100.0

H = 1000.0
G = 9.81

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ------------------------------------------------------------
# Dataset
# ------------------------------------------------------------
class DatasetSW(Dataset):
    def __init__(self, root):
        self.samples = []
        ic_dirs = sorted(glob.glob(os.path.join(root, "*")))

        for d in ic_dirs:
            files = sorted(glob.glob(os.path.join(d, "klein_step_*.npz")))
            for i in range(len(files)-ROLL_STEPS):
                self.samples.append((files[i:i+ROLL_STEPS+1], os.path.basename(d)))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        files, ic = self.samples[idx]

        seq = []
        for f in files:
            z = np.load(f)
            seq.append(np.stack([z["eta"], z["uc"], z["vc"]]))

        seq = np.stack(seq).astype(np.float32)

        return {
            "seq": torch.tensor(seq),
            "ic": ic,
        }

# ------------------------------------------------------------
# Model
# ------------------------------------------------------------
class Model(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3,64,3,padding=1), nn.GELU(),
            nn.Conv2d(64,64,3,padding=1), nn.GELU(),
            nn.Conv2d(64,64,3,padding=1), nn.GELU(),
            nn.Conv2d(64,3,3,padding=1)
        )

    def forward(self, x):
        return self.net(x)

# ------------------------------------------------------------
# Physics loss (single step)
# ------------------------------------------------------------
def physics_loss(xdot, x, colloc_bank, ic_key, macro_index):

    colloc = colloc_bank.sample_nearest_macro(ic_key, macro_index, 64)

    iy = torch.tensor(colloc["iy"], device=device)
    ix = torch.tensor(colloc["ix"], device=device)

    eta_t = xdot[0,iy,ix]
    u_t = xdot[1,iy,ix]
    v_t = xdot[2,iy,ix]

    eta = x[0,iy,ix]
    u = x[1,iy,ix]
    v = x[2,iy,ix]

    div = torch.tensor(colloc["div_fd"], device=device)
    etax = torch.tensor(colloc["etax_fd"], device=device)
    etay = torch.tensor(colloc["etay_fd"], device=device)
    f = torch.tensor(colloc["f"], device=device)

    # continuity
    L_wave = ((eta_t + H*div)**2).mean()

    # momentum
    Ru = u_t - f*v + G*etax
    Rv = v_t + f*u + G*etay
    L_mom = (Ru**2).mean() + (Rv**2).mean()

    # tendency match
    L_colloc = (
        (eta_t - torch.tensor(colloc["deta_dt_fd"], device=device))**2
    ).mean()

    return L_colloc, L_wave, L_mom

# ------------------------------------------------------------
# Train
# ------------------------------------------------------------
def train():

    dataset = DatasetSW(DATA_DIR)
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

    colloc_bank = CollocBank(COLLOC_DIR)

    model = Model().to(device)
    opt = torch.optim.Adam(model.parameters(), lr=LR)

    for ep in range(EPOCHS):

        t0 = time.time()

        for i,batch in enumerate(loader):

            seq = batch["seq"].to(device)   # [B, T, 3, ny, nx]
            ic = batch["ic"]

            B = seq.shape[0]

            loss_data = 0.0
            loss_colloc = 0.0
            loss_wave = 0.0
            loss_mom = 0.0

            x = seq[:,0]

            for k in range(ROLL_STEPS):

                xdot = model(x)
                x = x + DT * xdot

                # data loss
                loss_data += GAMMA**k * ((x - seq[:,k+1])**2).mean()

                # physics at EACH step
                for b in range(B):
                    Lc, Lw, Lm = physics_loss(
                        xdot[b], x[b],
                        colloc_bank,
                        ic[b],
                        k+1
                    )
                    loss_colloc += Lc
                    loss_wave += Lw
                    loss_mom += Lm

            loss = (
                loss_data
                + LAMBDA_COLLOC*loss_colloc
                + LAMBDA_WAVE*loss_wave
                + LAMBDA_MOM*loss_mom
            )

            opt.zero_grad()
            loss.backward()
            opt.step()

            if i % 20 == 0:
                print(f"[ep {ep} batch {i}] total={loss.item():.3e}")

        print(f"Epoch {ep} done in {time.time()-t0:.1f}s")

    torch.save(model.state_dict(), f"{CKPT_DIR}/model.pt")

# ------------------------------------------------------------
if __name__ == "__main__":
    train()

[CollocBank] loaded gauss_00 | samples=15616 | macro_groups=7
[CollocBank] loaded gauss_01 | samples=15616 | macro_groups=7
[CollocBank] loaded gauss_02 | samples=15616 | macro_groups=7
[CollocBank] loaded gauss_03 | samples=15616 | macro_groups=7
[CollocBank] loaded gauss_04 | samples=15616 | macro_groups=7
[CollocBank] loaded gauss_05 | samples=15616 | macro_groups=7
[CollocBank] loaded mix_00 | samples=15616 | macro_groups=7
[CollocBank] loaded mix_01 | samples=15616 | macro_groups=7
[CollocBank] loaded mix_02 | samples=15616 | macro_groups=7
[CollocBank] loaded mix_03 | samples=15616 | macro_groups=7
[CollocBank] loaded mix_04 | samples=15616 | macro_groups=7
[CollocBank] loaded mix_05 | samples=15616 | macro_groups=7
[CollocBank] loaded test_modon_00 | samples=15616 | macro_groups=7
[CollocBank] loaded test_modon_01 | samples=15616 | macro_groups=7
[CollocBank] loaded test_rh_00 | samples=15616 | macro_groups=7
[CollocBank] loaded test_rh_01 | samples=15616 | macro_groups=7
[Collo

# Continuous PINN with vorticity added to physics